# BEVFormer in Pure PyTorch: A Literate Architecture Walkthrough

This notebook builds the entire BEVFormer architecture used by this repository,
one tensor contract at a time, **without importing the `bevformer` package** —
every class and function below is copied verbatim from the corresponding file
under `bevformer/`, so this notebook is a second, from-scratch, teaching-oriented
construction of the exact same model, not a wrapper around it.

Each section follows the same three-cell pattern:

1. **Markdown** — the mathematical definition of the operation, with the tensor
   shapes it consumes and produces.
2. **Code** — the PyTorch implementation, with inline comments tracking the
   shape of every intermediate tensor.
3. **Probe** — a tiny synthetic input run through the just-defined module, with
   `assert`s that pin down its output contract (a runnable version of the
   markdown's claims).

## Symbols

| Symbol | Meaning | Default in this repo |
|---|---|---:|
| `B` | batch size | training dependent |
| `T` | temporal queue length | 4 |
| `N` | camera views | 6 |
| `C` | embedding / FPN channel width | 256 |
| `L` | FPN levels used by the BEV encoder | 4 |
| `K` | nuScenes detection classes | 10 |
| `H_bev, W_bev` | BEV grid height/width | 200 x 200 |
| `D` | pillar height samples per BEV cell (spatial cross-attn) | 4 |
| `Q` | object queries (decoder) | 900 |
| `M` | attention heads | 8 |
| `P` | learned sampling points per (head, level) in deformable attention | 4 |

## Architecture At A Glance

```text
                         ┌─────────────────────────────────────────────┐
                         │              queue of T frames               │
                         │      images[t]: [B,N,3,H,W], t=0..T-1        │
                         └───────────────────────┬───────────────────────┘
                                                  │  for t = 0 .. T-1
                     ┌────────────────────────────▼────────────────────────────┐
                     │  MultiViewImageBackbone (ResNet + DCN, stages 4-5)        │
                     │  [B,N,3,H,W] -> {stage3,stage4,stage5}                    │
                     └────────────────────────────┬────────────────────────────┘
                                                  │
                     ┌────────────────────────────▼────────────────────────────┐
                     │  ImageFPN: top-down 4-level pyramid                       │
                     │  {stage3,4,5} -> {p3,p4,p5,p6}, each [B,N,C,H_l,W_l]      │
                     └────────────────────────────┬────────────────────────────┘
                                                  │  mlvl_feats
     prev_bev (warped) ──────────────────────────►│
                     ┌────────────────────────────▼────────────────────────────┐
                     │  BEVFormerEncoder (this frame's layer stack)              │
                     │    per layer: TemporalSelfAttention -> SpatialCrossAttn   │
                     │               -> FFN                                     │
                     │  -> bev_embed: [B, H_bev*W_bev, C]                        │
                     └────────────────────────────┬────────────────────────────┘
                                                  │  t < T-1: no_grad, becomes next prev_bev
                                                  │  t = T-1: grad enabled, feeds the decoder
                     ┌────────────────────────────▼────────────────────────────┐
                     │  BEVFormerDecoder (object queries, Q learned queries)     │
                     │    per layer: self-attn(queries) -> deformable            │
                     │               cross-attn(queries, bev_embed) -> FFN       │
                     │  -> hidden_states: [num_dec_layers, B, Q, C]              │
                     └────────────────────────────┬────────────────────────────┘
                                                  │
                     ┌────────────────────────────▼────────────────────────────┐
                     │  BEVFormerHead: per-layer cls_branch + reg_branch         │
                     │  -> cls_scores [L,B,Q,K], bbox_preds [L,B,Q,10]           │
                     └────────────────────────────┬────────────────────────────┘
                                                  │
                     ┌────────────────────────────▼────────────────────────────┐
                     │  BEVFormerLoss: HungarianMatcher3D + focal + L1           │
                     └───────────────────────────────────────────────────────────┘
```

In [1]:
from __future__ import annotations

import math
from typing import Callable

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models
from torchvision.models import ResNet50_Weights, ResNet101_Weights
from torchvision.ops import DeformConv2d

torch.manual_seed(0)
SMOKE_MODE = True  # keep every probe tiny and CPU-friendly

## GridMask: Structured Occlusion Augmentation

GridMask (Chen et al., 2020) removes a periodic grid of square regions from the
image instead of random rectangles (Cutout) — the periodicity guarantees
coverage of the whole image regardless of where objects sit.

For a canvas of spacing $d$ and keep-ratio $r$, the removed stripe width is

$$l = \operatorname{clip}(\operatorname{round}(d \cdot r),\ 1,\ d-1)$$

A binary canvas 1.5x the image size is built, vertical and horizontal stripes
of width $l$ every $d$ pixels are zeroed (with a random phase offset and
optional rotation), then cropped back to the image size and optionally
inverted (`mode=1` keeps the grid *lines*, zeroing the *cells* — the common
setting). The final mask $M \in \{0,1\}^{H\times W}$ is broadcast over the
channel dimension:

$$\text{out} = \text{image} \odot M$$

Applied only during training (`self.training`), and only with probability
`probability` per forward call — so most iterations still see a clean image.

**Contract:** `images: [B*N, 3, H, W] -> [B*N, 3, H, W]` (shape-preserving).

In [2]:
class GridMask(nn.Module):
    def __init__(
        self,
        use_h: bool = True,
        use_w: bool = True,
        rotate: int = 1,
        offset: bool = False,
        ratio: float = 0.5,
        mode: int = 1,
        probability: float = 0.7,
    ) -> None:
        super().__init__()
        self.use_h = use_h
        self.use_w = use_w
        self.rotate = rotate
        self.offset = offset
        self.ratio = ratio
        self.mode = mode
        self.probability = probability

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if not self.training or np.random.rand() > self.probability:
            return images
        height, width = images.shape[-2:]
        canvas_height = int(1.5 * height)
        canvas_width = int(1.5 * width)
        spacing = np.random.randint(2, height)
        line_width = min(max(int(spacing * self.ratio + 0.5), 1), spacing - 1)
        mask = np.ones((canvas_height, canvas_width), dtype=np.float32)
        start_h = np.random.randint(spacing)
        start_w = np.random.randint(spacing)
        if self.use_h:
            for index in range(canvas_height // spacing):
                start = spacing * index + start_h
                mask[start : min(start + line_width, canvas_height), :] = 0
        if self.use_w:
            for index in range(canvas_width // spacing):
                start = spacing * index + start_w
                mask[:, start : min(start + line_width, canvas_width)] = 0

        rotation = np.random.randint(self.rotate) if self.rotate > 1 else 0
        mask = np.asarray(Image.fromarray(mask).rotate(rotation)).copy()
        top = (canvas_height - height) // 2
        left = (canvas_width - width) // 2
        mask = mask[top : top + height, left : left + width]
        mask_tensor = torch.as_tensor(mask, device=images.device, dtype=images.dtype)
        if self.mode == 1:
            mask_tensor = 1 - mask_tensor
        mask_tensor = mask_tensor.view(1, 1, height, width)
        if self.offset:
            offset_tensor = (
                torch.rand(1, images.shape[-3], height, width, device=images.device, dtype=images.dtype) * 2 - 1
            )
            return images * mask_tensor + offset_tensor * (1 - mask_tensor)
        return images * mask_tensor

In [3]:
# --- Probe: eval mode is a no-op; train mode with probability=1 always masks ---
grid_probe = GridMask(probability=1.0)
grid_images = torch.ones(2, 3, 3, 16, 24)          # [B*N=2, C=3, H=16, W=24]

grid_probe.eval()
grid_eval_out = grid_probe(grid_images)
assert torch.equal(grid_eval_out, grid_images)      # eval() -> identity

grid_probe.train()
grid_train_out = grid_probe(grid_images)
assert grid_train_out.shape == grid_images.shape    # shape-preserving
assert not torch.equal(grid_train_out, grid_images) # some cells actually zeroed
print("GridMask ok:", grid_train_out.shape, "fraction kept:", grid_train_out.mean().item())

GridMask ok: torch.Size([2, 3, 3, 16, 24]) fraction kept: 0.75


## Deformable Convolution: Learned Sampling Offsets In The Backbone

A standard $k\times k$ convolution samples a fixed grid of offsets
$\{\Delta p_n\}_{n=1}^{k^2}$ (e.g. $\{(-1,-1),\dots,(1,1)\}$ for $k=3$) around
each output location $p_0$:

$$y(p_0) = \sum_{n=1}^{k^2} w_n \cdot x(p_0 + \Delta p_n)$$

A **deformable** convolution (Dai et al., 2017) instead *learns* an additional
per-location offset $\Delta m_n \in \mathbb{R}^2$ from the input itself, via a
parallel plain convolution (`offset_conv`) with $2k^2$ output channels (one
$(dx, dy)$ pair per kernel tap):

$$y(p_0) = \sum_{n=1}^{k^2} w_n \cdot x(p_0 + \Delta p_n + \Delta m_n(p_0))$$

Since $\Delta m_n(p_0)$ is a continuous offset, $x$ is sampled with bilinear
interpolation — this is exactly `torchvision.ops.DeformConv2d`. Official
BEVFormer applies this to the last two ResNet stages (`stage4`, `stage5`),
letting the receptive field adapt to object shape/orientation instead of
being a fixed square.

**Initialization matters:** `offset_conv`'s weight and bias start at zero, so
$\Delta m_n(p_0) = 0$ everywhere at step 0 — the deformable conv is *exactly*
a normal conv until training nudges the offsets away from zero. The dense
conv's pretrained weights are copied into `deform_conv` so the backbone starts
from ImageNet features unchanged.

**Contract:** `x: [B, C_in, H, W] -> [B, C_out, H, W]` (shape-preserving for
stride 1, matches the wrapped `nn.Conv2d`'s output shape otherwise).

In [4]:
class DeformConv2dPack(nn.Module):
    """Drop-in replacement for a ResNet 3x3 conv using learned offsets."""

    def __init__(self, conv: nn.Conv2d) -> None:
        super().__init__()
        kernel_h, kernel_w = conv.kernel_size
        offset_channels = 2 * kernel_h * kernel_w
        self.offset_conv = nn.Conv2d(
            in_channels=conv.in_channels,
            out_channels=offset_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            bias=True,
        )
        self.deform_conv = DeformConv2d(
            in_channels=conv.in_channels,
            out_channels=conv.out_channels,
            kernel_size=kernel_h,
            stride=conv.stride[0],
            padding=conv.padding[0],
            dilation=conv.dilation[0],
            groups=conv.groups,
            bias=conv.bias is not None,
        )
        with torch.no_grad():
            self.deform_conv.weight.copy_(conv.weight)
            if conv.bias is not None and self.deform_conv.bias is not None:
                self.deform_conv.bias.copy_(conv.bias)
            nn.init.zeros_(self.offset_conv.weight)
            nn.init.zeros_(self.offset_conv.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        offset = self.offset_conv(x)
        return self.deform_conv(x, offset)

In [5]:
# --- Probe: zero-initialized offsets => output equals a plain conv's output ---
plain_conv = nn.Conv2d(4, 6, kernel_size=3, padding=1)
deform_pack = DeformConv2dPack(plain_conv)

deform_input = torch.randn(1, 4, 8, 8)
with torch.no_grad():
    plain_out = plain_conv(deform_input)
    deform_out = deform_pack(deform_input)
torch.testing.assert_close(deform_out, plain_out, atol=1e-5, rtol=1e-5)
print("DeformConv2dPack ok: matches plain conv at zero offsets, shape", deform_out.shape)

DeformConv2dPack ok: matches plain conv at zero offsets, shape torch.Size([1, 6, 8, 8])


## MultiViewImageBackbone: One Shared ResNet Over All Cameras

All $N$ cameras are processed by **one** ResNet with **shared weights** — the
camera axis is folded into the batch axis before the stem, and restored after
each stage, so BEVFormer's per-camera features cost exactly one ResNet forward
pass over $B \cdot N$ images:

$$[B, N, 3, H, W] \xrightarrow{\text{reshape}} [B{\cdot}N, 3, H, W]
\xrightarrow{\text{ResNet}} \{C_3, C_4, C_5\}
\xrightarrow{\text{reshape}} \{[B,N,C_l,H_l,W_l]\}_{l=3}^{5}$$

`stage4`'s and `stage5`'s $3\times3$ convolutions (`conv2` of every residual
block) are replaced with `DeformConv2dPack` from the previous section —
this is the "ResNet + DCN" backbone used by the official (non-VoVNet)
BEVFormer configs. `frozen_stages` optionally freezes the stem and the first
`frozen_stages` residual stages (both their gradients and their BatchNorm
running statistics via `norm_eval`), standard practice when fine-tuning an
ImageNet backbone on a comparatively small detection dataset.

In [6]:
_WEIGHTS = {
    "resnet50": (models.resnet50, ResNet50_Weights.IMAGENET1K_V2),
    "resnet101": (models.resnet101, ResNet101_Weights.IMAGENET1K_V2),
}

In [7]:
class MultiViewImageBackbone(nn.Module):
    """Apply a torchvision ResNet backbone to all camera views.

    Input: images [B, N, 3, H, W]
    Output: dict[str, Tensor] with "stage3"/"stage4"/"stage5", each [B, N, C, H_l, W_l]
    """

    def __init__(
        self,
        variant: str = "resnet50",
        pretrained: bool = True,
        frozen_stages: int = 1,
        norm_eval: bool = True,
    ) -> None:
        super().__init__()
        if variant not in _WEIGHTS:
            raise ValueError(f"Unsupported backbone variant: {variant}")
        constructor, weights_enum = _WEIGHTS[variant]
        backbone = constructor(weights=weights_enum if pretrained else None)

        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.stage2 = backbone.layer1
        self.stage3 = backbone.layer2
        self.stage4 = backbone.layer3
        self.stage5 = backbone.layer4
        self._convert_stage_to_deformable(self.stage4)
        self._convert_stage_to_deformable(self.stage5)

        self.frozen_stages = frozen_stages
        self.norm_eval = norm_eval
        self._freeze_stages()

    @staticmethod
    def _convert_stage_to_deformable(stage: nn.Sequential) -> None:
        for block in stage:
            if hasattr(block, "conv2") and isinstance(block.conv2, nn.Conv2d):
                block.conv2 = DeformConv2dPack(block.conv2)

    def _freeze_stages(self) -> None:
        if self.frozen_stages >= 0:
            self.stem.eval()
            for param in self.stem.parameters():
                param.requires_grad = False
        stages = [self.stage2, self.stage3, self.stage4, self.stage5]
        for idx, stage in enumerate(stages, start=1):
            if self.frozen_stages >= idx:
                stage.eval()
                for param in stage.parameters():
                    param.requires_grad = False

    def train(self, mode: bool = True) -> "MultiViewImageBackbone":
        super().train(mode)
        self._freeze_stages()
        if mode and self.norm_eval:
            for module in self.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images: torch.Tensor) -> dict[str, torch.Tensor]:
        batch, num_cams, channels, height, width = images.shape
        flat = images.reshape(batch * num_cams, channels, height, width)

        x = self.stem(flat)
        stage2 = self.stage2(x)
        stage3 = self.stage3(stage2)
        stage4 = self.stage4(stage3)
        stage5 = self.stage5(stage4)

        features = {"stage3": stage3, "stage4": stage4, "stage5": stage5}
        multi_view = {}
        for name, feat in features.items():
            _, out_channels, out_height, out_width = feat.shape
            multi_view[name] = feat.reshape(batch, num_cams, out_channels, out_height, out_width)
        return multi_view

In [8]:
# --- Probe: tiny resnet50 backbone, no pretrained download, on 2 cams ---
backbone_probe = MultiViewImageBackbone(variant="resnet50", pretrained=False, frozen_stages=1).eval()
backbone_images = torch.randn(1, 2, 3, 64, 96)  # B=1, N=2 cams, 64x96 images

with torch.inference_mode():
    backbone_feats = backbone_probe(backbone_images)

for name, expected_c, expected_hw in [("stage3", 512, (8, 12)), ("stage4", 1024, (4, 6)), ("stage5", 2048, (2, 3))]:
    feat = backbone_feats[name]
    assert feat.shape == (1, 2, expected_c, *expected_hw), (name, feat.shape)
    print(f"{name}: {tuple(feat.shape)}   # [B, N, C_{name[-1]}, H_l, W_l]")

stage3: (1, 2, 512, 8, 12)   # [B, N, C_3, H_l, W_l]
stage4: (1, 2, 1024, 4, 6)   # [B, N, C_4, H_l, W_l]
stage5: (1, 2, 2048, 2, 3)   # [B, N, C_5, H_l, W_l]


## ImageFPN: A Four-Level Top-Down Feature Pyramid

Deep stages ($C_5$) carry strong semantics but coarse spatial resolution;
shallow stages ($C_3$) carry the opposite trade-off. A Feature Pyramid
Network (Lin et al., 2017) fuses them top-down so *every* output level has
both:

$$L_l = \operatorname{Conv}_{1\times1}(C_l) \quad\text{(unify channel width to } C\text{)}$$
$$T_l = L_l + \operatorname{upsample}_{\text{nearest}}(T_{l+1}), \qquad T_5 := L_5$$
$$P_l = \operatorname{Conv}_{3\times3}(T_l) \qquad l = 3,4,5$$
$$P_6 = \operatorname{Conv}_{3\times3,\ \text{stride }2}(P_5)$$

$P_6$ gives the encoder access to an even coarser level (useful for large
objects / long-range BEV cells) without another backbone stage. All four
levels share the same channel width $C$, which is exactly what
`MultiScaleDeformableAttention` needs later (`num_levels=4`).

**Contract:** `{stage3,4,5}: [B,N,C_l,H_l,W_l] -> {p3,p4,p5,p6}: [B,N,C,H_l,W_l]`,
with $H_{l+1} = \lceil H_l / 2 \rceil$ (standard stride-2 pyramid spacing).

In [9]:
class ImageFPN(nn.Module):
    """Build a top-down feature pyramid from multi-view backbone features."""

    def __init__(
        self,
        in_channels: Iterable[int] = (512, 1024, 2048),
        out_channels: int = 256,
        out_names: Iterable[str] = ("p3", "p4", "p5", "p6"),
    ) -> None:
        super().__init__()
        self.out_names = list(out_names)
        self.lateral_convs = nn.ModuleList([nn.Conv2d(ch, out_channels, kernel_size=1) for ch in in_channels])
        self.output_convs = nn.ModuleList(
            [nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1) for _ in in_channels]
        )
        self.extra_conv = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1)

    def forward(self, features: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        stage_names = sorted(features.keys())
        batch, num_cams = next(iter(features.values())).shape[:2]
        flat_feats = [
            features[name].reshape(batch * num_cams, *features[name].shape[2:]) for name in stage_names
        ]

        laterals = [conv(feat) for conv, feat in zip(self.lateral_convs, flat_feats)]
        for idx in range(len(laterals) - 1, 0, -1):
            laterals[idx - 1] = laterals[idx - 1] + F.interpolate(
                laterals[idx], size=laterals[idx - 1].shape[-2:], mode="nearest"
            )

        outputs = [conv(feat) for conv, feat in zip(self.output_convs, laterals)]
        outputs.append(self.extra_conv(outputs[-1]))

        pyramid: dict[str, torch.Tensor] = {}
        for out_name, feat in zip(self.out_names, outputs):
            pyramid[out_name] = feat.reshape(batch, num_cams, feat.shape[1], feat.shape[2], feat.shape[3])
        return pyramid

In [10]:
# --- Probe: reuse the backbone features from the previous probe ---
fpn_probe = ImageFPN(in_channels=(512, 1024, 2048), out_channels=32).eval()
with torch.inference_mode():
    pyramid = fpn_probe(backbone_feats)

expected_shapes = {"p3": (8, 12), "p4": (4, 6), "p5": (2, 3), "p6": (1, 2)}
for name, hw in expected_shapes.items():
    feat = pyramid[name]
    assert feat.shape == (1, 2, 32, *hw), (name, feat.shape)
    print(f"{name}: {tuple(feat.shape)}   # [B, N, C=32, H_l, W_l]")

p3: (1, 2, 32, 8, 12)   # [B, N, C=32, H_l, W_l]
p4: (1, 2, 32, 4, 6)   # [B, N, C=32, H_l, W_l]
p5: (1, 2, 32, 2, 3)   # [B, N, C=32, H_l, W_l]
p6: (1, 2, 32, 1, 2)   # [B, N, C=32, H_l, W_l]


## Geometry Helpers: Normalized Parameters <-> Metric Coordinates

BEVFormer keeps every learned spatial quantity (BEV cell centers, pillar
heights, object-query reference points) as a **normalized** coordinate in
$[0,1]$, decoupling the network's parameterization from the metric point
cloud range `pc_range = (x_{\min},y_{\min},z_{\min},x_{\max},y_{\max},z_{\max})`.
Two conversions recur everywhere:

**Denormalize** (normalized -> metric), used whenever a normalized point must
be projected into a camera or reported as a final box coordinate:

$$p_{\text{metric}} = p_{\text{norm}} \odot (p_{\max} - p_{\min}) + p_{\min}$$

**Inverse sigmoid** (probability -> logit), used to compose a *residual*
regression output with an existing sigmoid-bounded reference point before
re-applying sigmoid — this is how the decoder refines box centers layer by
layer without ever leaving $[0,1]$:

$$\operatorname{logit}(x) = \ln\frac{x}{1-x}, \qquad x \text{ clamped to } [\epsilon, 1-\epsilon]$$

so that $\operatorname{sigmoid}(\operatorname{logit}(r) + \Delta) $ is a new
reference point nudged by the network's predicted offset $\Delta$.

In [11]:
def inverse_sigmoid(x: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    x = x.clamp(min=eps, max=1 - eps)
    return torch.log(x / (1 - x))


def denormalize_reference_points(
    reference_points: torch.Tensor,
    pc_range: tuple[float, float, float, float, float, float],
) -> torch.Tensor:
    pc_range_t = torch.as_tensor(pc_range, dtype=reference_points.dtype, device=reference_points.device)
    xyz_min = pc_range_t[:3]
    xyz_max = pc_range_t[3:]
    return reference_points * (xyz_max - xyz_min) + xyz_min

In [12]:
# --- Probe: round trip, and corner/midpoint mapping ---
probe_x = torch.tensor([0.1, 0.5, 0.9])
torch.testing.assert_close(torch.sigmoid(inverse_sigmoid(probe_x)), probe_x, atol=1e-4, rtol=1e-4)

probe_pc_range = (-10.0, -20.0, -2.0, 10.0, 20.0, 2.0)
probe_corners = torch.tensor([[0.0, 0.0, 0.0], [1.0, 1.0, 1.0], [0.5, 0.5, 0.5]])
probe_metric = denormalize_reference_points(probe_corners, probe_pc_range)
print("normalized -> metric:")
for norm_pt, metric_pt in zip(probe_corners, probe_metric):
    print(f"  {norm_pt.tolist()} -> {metric_pt.tolist()}")
assert torch.allclose(probe_metric[2], torch.zeros(3))  # 0.5 -> range midpoint

normalized -> metric:
  [0.0, 0.0, 0.0] -> [-10.0, -20.0, -2.0]
  [1.0, 1.0, 1.0] -> [10.0, 20.0, 2.0]
  [0.5, 0.5, 0.5] -> [0.0, 0.0, 0.0]


## The BEV Grid And Its Vertical Pillars

BEVFormer represents the scene as an $H_{\text{bev}} \times W_{\text{bev}}$
grid of learned queries laid out over the ground plane. Cell $(i, j)$'s
normalized center (row-major, $i$ = row = $y$, $j$ = col = $x$) is

$$q_{i,j} = \left(\frac{j + 0.5}{W_{\text{bev}}},\ \frac{i + 0.5}{H_{\text{bev}}}\right) \in [0,1]^2$$

For **spatial cross-attention**, each flat BEV query additionally needs a
guess at *where it is in 3D*, so it can be projected into the cameras. Since a
single ground-plane point is ambiguous in height, BEVFormer samples $D$
points stacked vertically per cell — a "pillar" — evenly spaced in $z$
between $z_{\min}$ and $z_{\max}$:

$$z_d = z_{\min} + \frac{d}{D-1}(z_{\max}-z_{\min}), \qquad d = 0,\dots,D-1$$

giving pillar reference points $(q_{i,j,x},\, q_{i,j,y},\, \tilde z_d)$ for
every $(cell, height)$ pair, all still normalized to $[0,1]^3$.

**Contract:** `get_bev_grid_points_2d(H,W) -> [H*W, 2]`;
`get_pillar_reference_points_3d(H,W,pc_range,D) -> [D, H*W, 3]`.

In [13]:
def get_bev_grid_points_2d(bev_h: int, bev_w: int) -> torch.Tensor:
    """Normalized [0,1] xy cell centers of the BEV grid, row-major order."""
    ys, xs = torch.meshgrid(
        (torch.arange(bev_h, dtype=torch.float32) + 0.5) / bev_h,
        (torch.arange(bev_w, dtype=torch.float32) + 0.5) / bev_w,
        indexing="ij",
    )
    return torch.stack([xs.reshape(-1), ys.reshape(-1)], dim=-1)


def get_pillar_reference_points_3d(
    bev_h: int,
    bev_w: int,
    pc_range: tuple[float, float, float, float, float, float],
    num_points_in_pillar: int,
) -> torch.Tensor:
    """Normalized [0,1] xyz points: `num_points_in_pillar` heights per BEV cell."""
    grid_xy = get_bev_grid_points_2d(bev_h, bev_w)  # [Q, 2]
    z_min, z_max = pc_range[2], pc_range[5]
    heights_metric = torch.linspace(z_min, z_max, num_points_in_pillar)
    heights_norm = (heights_metric - z_min) / (z_max - z_min)

    num_query = grid_xy.shape[0]
    xy = grid_xy.unsqueeze(0).expand(num_points_in_pillar, num_query, 2)
    z = heights_norm.view(num_points_in_pillar, 1, 1).expand(num_points_in_pillar, num_query, 1)
    return torch.cat([xy, z], dim=-1)

In [14]:
# --- Probe: a tiny 4x4 BEV grid with 3 pillar heights ---
bev_grid = get_bev_grid_points_2d(bev_h=4, bev_w=4)
assert bev_grid.shape == (16, 2)
assert bev_grid[0, 1] == bev_grid[1, 1]  # row 0's two first cells share y
assert bev_grid[0, 1] != bev_grid[4, 1]  # row 0 vs row 1 differ in y

pillar_points = get_pillar_reference_points_3d(bev_h=4, bev_w=4, pc_range=probe_pc_range, num_points_in_pillar=3)
assert pillar_points.shape == (3, 16, 3)               # [D, H*W, 3]
assert len(torch.unique(pillar_points[:, 0, 2])) == 3  # 3 distinct heights
print("BEV grid:", bev_grid.shape, " pillar points:", pillar_points.shape)

BEV grid: torch.Size([16, 2])  pillar points: torch.Size([3, 16, 3])


## Projecting Pillar Points Into Every Camera

Each pillar point $p = (x,y,z)$ (in the LiDAR/ego reference frame) is
projected into camera $n$ with that camera's $4{\times}4$ `lidar2img` matrix
$P_n$ (intrinsics $\times$ extrinsics, precomputed once per sample in the
data pipeline from calibrated-sensor + ego-pose records):

$$p_h = [x,\,y,\,z,\,1]^\top, \qquad
s = P_n\, p_h = [s_x,\, s_y,\, d,\, s_w]^\top$$

$d$ (the third homogeneous component) is the point's depth in the camera
frame. Perspective division gives the pixel location, normalized to
$[0,1]$ by the image size $(H_{\text{img}}, W_{\text{img}})$:

$$u = \frac{s_x}{d\,W_{\text{img}}}, \qquad v = \frac{s_y}{d\,H_{\text{img}}}$$

A projection is **valid** only if the point is in front of the camera and
lands inside the image:

$$\text{valid} = (d > \epsilon) \ \wedge\ (0 \le u \le 1) \ \wedge\ (0 \le v \le 1)$$

Every pillar point is projected into every camera independently — a BEV cell
near the vehicle's side will typically be valid in exactly one camera; a cell
far away or between camera frustums may be valid in zero or (at overlaps) two.

**Contract:** `reference_points_3d: [D,Q,3]` (normalized) `-> reference_points_cam:
[N_{\text{cam}}, B, Q, D, 2]` (normalized image xy), `bev_mask: [N_{\text{cam}}, B, Q, D]` (bool).

In [15]:
def project_pillar_points_to_cameras(
    reference_points_3d: torch.Tensor,
    pc_range: tuple[float, float, float, float, float, float],
    img_metas: list[dict],
) -> tuple[torch.Tensor, torch.Tensor]:
    """Projects normalized [0,1] pillar points into every camera.

    Args:
        reference_points_3d: [D, Q, 3] normalized xyz.
        pc_range: [xmin, ymin, zmin, xmax, ymax, zmax].
        img_metas: list of length B, each with "lidar2img" (list of num_cam
            4x4 matrices) and "image_size" (height, width).

    Returns:
        reference_points_cam: [num_cam, B, Q, D, 2] normalized [0,1] image xy.
        bev_mask: [num_cam, B, Q, D] bool validity.
    """
    device = reference_points_3d.device
    dtype = torch.float32
    num_points_in_pillar, num_query, _ = reference_points_3d.shape
    batch = len(img_metas)
    num_cam = len(img_metas[0]["lidar2img"])

    xyz_min = torch.tensor(pc_range[:3], dtype=dtype, device=device)
    xyz_max = torch.tensor(pc_range[3:], dtype=dtype, device=device)
    points_metric = reference_points_3d.to(dtype) * (xyz_max - xyz_min) + xyz_min  # [D, Q, 3]
    points_homo = torch.cat([points_metric, torch.ones_like(points_metric[..., :1])], dim=-1)
    points_flat = points_homo.reshape(num_points_in_pillar * num_query, 4)  # [D*Q, 4]

    reference_points_cam = torch.zeros(num_cam, batch, num_query, num_points_in_pillar, 2, dtype=dtype, device=device)
    bev_mask = torch.zeros(num_cam, batch, num_query, num_points_in_pillar, dtype=torch.bool, device=device)

    for b, meta in enumerate(img_metas):
        lidar2img = torch.stack(
            [torch.as_tensor(m, dtype=dtype, device=device) for m in meta["lidar2img"]], dim=0
        )  # [num_cam, 4, 4]
        image_h, image_w = meta["image_size"]

        proj = torch.einsum("cij,pj->cpi", lidar2img, points_flat)  # [num_cam, D*Q, 4]
        depth = proj[..., 2]
        xy = proj[..., :2] / depth.clamp(min=1e-5).unsqueeze(-1)
        norm_x = xy[..., 0] / image_w
        norm_y = xy[..., 1] / image_h
        valid = (
            (depth > 1e-5)
            & (norm_x >= 0.0)
            & (norm_x <= 1.0)
            & (norm_y >= 0.0)
            & (norm_y <= 1.0)
        )

        points_cam = torch.stack([norm_x, norm_y], dim=-1).reshape(
            num_cam, num_points_in_pillar, num_query, 2
        )
        reference_points_cam[:, b] = points_cam.permute(0, 2, 1, 3)
        bev_mask[:, b] = valid.reshape(num_cam, num_points_in_pillar, num_query).permute(0, 2, 1)

    return reference_points_cam, bev_mask

In [16]:
# --- Probe: identity camera (lidar frame == camera frame, fx=fy=1,cx=cy=0) ---
def identity_lidar2img():
    return np.eye(4, dtype=np.float32)

probe_img_metas = [{"lidar2img": [identity_lidar2img(), identity_lidar2img()], "image_size": (20, 20)}]
proj_ref_cam, proj_mask = project_pillar_points_to_cameras(pillar_points, probe_pc_range, probe_img_metas)

assert proj_ref_cam.shape == (2, 1, 16, 3, 2)  # [N_cam, B, Q, D, 2]
assert proj_mask.shape == (2, 1, 16, 3)        # [N_cam, B, Q, D]
assert proj_mask.dtype == torch.bool
print("reference_points_cam:", proj_ref_cam.shape, " bev_mask:", proj_mask.shape,
      " valid fraction:", proj_mask.float().mean().item())

reference_points_cam: torch.Size([2, 1, 16, 3, 2])  bev_mask: torch.Size([2, 1, 16, 3])  valid fraction: 0.0833333358168602


## Multi-Scale Deformable Attention: The Core Operator

Standard attention lets every query attend to *every* key — for a BEV grid
of $200{\times}200{=}40{,}000$ queries against image feature maps with
tens of thousands of pixels each, that's computationally infeasible. Deformable
attention (Zhu et al., *Deformable DETR*, 2021) fixes this by having each
query attend to only a **small, learned set of sampling points** near a given
reference location, across multiple feature-map resolutions ("levels") at once.

For query $q$ with feature vector $z_q \in \mathbb{R}^C$, reference point
$r_q \in [0,1]^2$, attention head $m = 1,\dots,M$, level $l = 1,\dots,L$, and
sample index $k = 1,\dots,K$ (this repo generalizes the reference point to
vary per (level, point) too — see below):

**1. Learned sampling offsets**, predicted directly from the query feature:

$$\Delta p_{mlk} = \operatorname{Linear}_{\text{offset}}(z_q) \in \mathbb{R}^2$$

**2. Sampling location** — the reference point nudged by that offset, in
level $l$'s own pixel grid (offsets are divided by level $l$'s $(W_l, H_l)$
so a fixed-magnitude learned offset covers proportionally the same distance
at every resolution):

$$p_{mlk} = r_q + \frac{\Delta p_{mlk}}{(W_l, H_l)}$$

**3. Attention weights**, also predicted from the query and normalized by
softmax over all $L{\times}K$ (level, point) pairs *for a given head*:

$$A_{mlk} = \operatorname{softmax}_{l,k}\big(\operatorname{Linear}_{\text{attn}}(z_q)\big)_{mlk}, \qquad \sum_{l,k} A_{mlk} = 1$$

**4. Output** — bilinearly sample the (linearly projected) value feature map
at each location and combine with the attention weights, summed over heads:

$$\text{out}_q = \operatorname{Linear}_{\text{out}}\Big(\underset{m}{\Big\Vert} \sum_{l=1}^{L}\sum_{k=1}^{K} A_{mlk}\cdot \operatorname{BilinearSample}\big(V_{ml},\ p_{mlk}\big)\Big)$$

where $\Vert$ concatenates the $M$ heads back into $C$ channels and
$\operatorname{BilinearSample}$ is exactly `F.grid_sample` (this repo's
pure-PyTorch stand-in for the official custom CUDA kernel — mathematically
identical, just implemented with a general-purpose op instead of a
hand-fused one).

**Why this one module powers both BEVFormer attentions:** letting the
reference point vary per (level, point) — rather than being one point shared
across all levels, as in vanilla Deformable DETR — means the *same* class can
serve as:
- **Spatial cross-attention**: level = FPN pyramid level, point = pillar
  height sample (a *different* 2D location per point, since each pillar
  height projects differently into the camera).
- **Temporal self-attention**: level = {previous BEV, current BEV} (two
  pseudo-levels of the *same* spatial grid, same reference point for both).
- A vanilla Deformable-DETR decoder cross-attention (used in this repo's own
  object-query decoder): level = FPN/BEV level, point = ordinary learned
  offset sample, reference point shared across levels.

An optional `point_mask` biases specific (level, point) pre-softmax logits by
a large negative constant ($-10^4$, not $-\infty$, to avoid NaNs when an
entire row is masked) — this is how spatial cross-attention excludes pillar
points that projected outside a given camera's image.

**Contract:** `query:[B,Q,C]`, `reference_points:[B,Q,L,K,2]`, `value:[B,S,C]`
(S = sum of $H_l W_l$ over levels) `-> [B,Q,C]`.

In [17]:
_MASKED_LOGIT = -1e4

In [18]:
class MultiScaleDeformableAttention(nn.Module):
    def __init__(self, embed_dims: int, num_heads: int, num_levels: int, num_points: int) -> None:
        super().__init__()
        if embed_dims % num_heads != 0:
            raise ValueError("embed_dims must be divisible by num_heads")
        self.embed_dims = embed_dims
        self.num_heads = num_heads
        self.num_levels = num_levels
        self.num_points = num_points
        self.head_dim = embed_dims // num_heads

        self.sampling_offsets = nn.Linear(embed_dims, num_heads * num_levels * num_points * 2)
        self.attention_weights = nn.Linear(embed_dims, num_heads * num_levels * num_points)
        self.value_proj = nn.Linear(embed_dims, embed_dims)
        self.output_proj = nn.Linear(embed_dims, embed_dims)

        nn.init.zeros_(self.sampling_offsets.weight)
        nn.init.zeros_(self.sampling_offsets.bias)
        nn.init.zeros_(self.attention_weights.weight)
        nn.init.zeros_(self.attention_weights.bias)

    def forward(
        self,
        query: torch.Tensor,
        reference_points: torch.Tensor,
        value: torch.Tensor,
        spatial_shapes: list[tuple[int, int]],
        point_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            query: [B, Q, C]
            reference_points: [B, Q, num_levels, num_points, 2] normalized [0,1] xy.
            value: [B, S, C] where S = sum(H_l * W_l).
            spatial_shapes: list of (H, W) per level, in the order `value` was flattened.
            point_mask: optional [B, Q, num_levels, num_points] bool, True = invalid/masked.
        Returns:
            [B, Q, C]
        """
        batch, num_query = query.shape[:2]
        num_value = value.shape[1]

        value_proj = self.value_proj(value).view(batch, num_value, self.num_heads, self.head_dim)

        offsets = self.sampling_offsets(query).view(
            batch, num_query, self.num_heads, self.num_levels, self.num_points, 2
        )
        raw_weights = self.attention_weights(query).view(
            batch, num_query, self.num_heads, self.num_levels * self.num_points
        )
        if point_mask is not None:
            mask = point_mask[:, :, None, :, :].expand(
                batch, num_query, self.num_heads, self.num_levels, self.num_points
            ).reshape(batch, num_query, self.num_heads, self.num_levels * self.num_points)
            raw_weights = raw_weights.masked_fill(mask, _MASKED_LOGIT)
        attention_weights = F.softmax(raw_weights, dim=-1).view(
            batch, num_query, self.num_heads, self.num_levels, self.num_points
        )

        offset_normalizer = torch.tensor(
            [[w, h] for h, w in spatial_shapes], dtype=query.dtype, device=query.device
        )  # [num_levels, 2]
        sampling_locations = (
            reference_points[:, :, None, :, :, :]
            + offsets / offset_normalizer[None, None, None, :, None, :]
        )  # [B, Q, num_heads, num_levels, num_points, 2]

        output = _multi_scale_deformable_attention_core(
            value_proj, spatial_shapes, sampling_locations, attention_weights
        )
        return self.output_proj(output)

In [19]:
def _multi_scale_deformable_attention_core(
    value: torch.Tensor,
    spatial_shapes: list[tuple[int, int]],
    sampling_locations: torch.Tensor,
    attention_weights: torch.Tensor,
) -> torch.Tensor:
    batch, _, num_heads, head_dim = value.shape
    _, num_query, _, num_levels, num_points, _ = sampling_locations.shape

    value_list = value.split([h * w for h, w in spatial_shapes], dim=1)
    sampling_grids = 2 * sampling_locations - 1  # [0,1] -> [-1,1]

    sampled_per_level = []
    for level, (h, w) in enumerate(spatial_shapes):
        # [B, H*W, heads, head_dim] -> [B*heads, head_dim, H, W]
        value_level = (
            value_list[level].flatten(2).transpose(1, 2).reshape(batch * num_heads, head_dim, h, w)
        )
        # [B, Q, heads, points, 2] -> [B*heads, Q, points, 2]
        grid_level = (
            sampling_grids[:, :, :, level].transpose(1, 2).reshape(batch * num_heads, num_query, num_points, 2)
        )
        sampled = F.grid_sample(
            value_level, grid_level, mode="bilinear", padding_mode="zeros", align_corners=False
        )  # [B*heads, head_dim, Q, points]
        sampled_per_level.append(sampled)

    # [B*heads, head_dim, Q, levels, points]
    sampled = torch.stack(sampled_per_level, dim=-2)
    weights = attention_weights.transpose(1, 2).reshape(batch * num_heads, 1, num_query, num_levels, num_points)
    output = (sampled * weights).sum(dim=(-1, -2))  # [B*heads, head_dim, Q]
    output = output.view(batch, num_heads * head_dim, num_query).transpose(1, 2)  # [B, Q, C]
    return output

In [20]:
# --- Probe: shape + gradient flow, 2 levels, 3 points, 4 heads ---
attn_probe = MultiScaleDeformableAttention(embed_dims=16, num_heads=4, num_levels=2, num_points=3)
attn_spatial_shapes = [(4, 4), (2, 2)]
attn_value = torch.randn(1, sum(h * w for h, w in attn_spatial_shapes), 16, requires_grad=True)
attn_query = torch.randn(1, 5, 16)
attn_reference_points = torch.rand(1, 5, 2, 3, 2)  # [B, Q, L=2, K=3, 2]

attn_out = attn_probe(attn_query, attn_reference_points, attn_value, attn_spatial_shapes)
assert attn_out.shape == (1, 5, 16)
attn_out.sum().backward()
assert attn_value.grad is not None and torch.isfinite(attn_value.grad).all()
print("MultiScaleDeformableAttention ok:", attn_out.shape, " gradient reaches value:", attn_value.grad.abs().sum().item() > 0)

MultiScaleDeformableAttention ok: torch.Size([1, 5, 16])  gradient reaches value: True


## Spatial Cross-Attention: Lifting Images Into BEV

Spatial cross-attention (SCA) answers: *for this BEV query, what do the
cameras that can see it look like?* For BEV query $q$ (one grid cell) and
camera $n$, it runs `MultiScaleDeformableAttention` with:

- **value** = camera $n$'s multi-level FPN features, flattened: $L{=}4$ levels.
- **reference points** = the $D$ pillar heights' 2D projections into camera
  $n$ (`reference_points_cam[n]`), broadcast across the 4 FPN levels — so
  `num_points := D` in the deformable-attention sense (each pillar height is
  one learned-offset sampling point, not a fixed point).
- **point_mask** = `~bev_mask[n]` — pillar points that didn't project inside
  camera $n$'s image contribute nothing to that camera's softmax.

The per-camera outputs are combined by a simple **indicator-weighted
average** over the cameras that see the query at all (this repo's
simplification of official BEVFormer's dynamic per-camera query gathering —
mathematically equivalent, just not index-gathered for simplicity):

$$\text{SCA}(q) = \frac{\sum_{n=1}^{N} \mathbb{1}[\exists d:\ \text{bev\_mask}[n,q,d]] \cdot \text{DeformAttn}_n(q)}{\max\Big(1,\ \sum_{n=1}^{N} \mathbb{1}[\exists d:\ \text{bev\_mask}[n,q,d]]\Big)}$$

A query invisible to every camera (e.g. a BEV cell behind the vehicle with no
rear camera coverage in a partial rig) gets a well-defined output of exactly
$\vec 0$, not NaN.

**Contract:** `query:[B,Q,C]`, `mlvl_feats: List[[B,N,C,H_l,W_l]]`,
`reference_points_cam:[N,B,Q,D,2]`, `bev_mask:[N,B,Q,D]` `-> [B,Q,C]`.

In [21]:
class SpatialCrossAttention(nn.Module):
    def __init__(
        self,
        embed_dims: int,
        num_cams: int,
        num_levels: int,
        num_points_in_pillar: int,
        num_heads: int = 8,
    ) -> None:
        super().__init__()
        self.num_cams = num_cams
        self.num_levels = num_levels
        self.deform_attn = MultiScaleDeformableAttention(
            embed_dims, num_heads, num_levels, num_points_in_pillar
        )

    def forward(
        self,
        query: torch.Tensor,
        mlvl_feats: list[torch.Tensor],
        reference_points_cam: torch.Tensor,
        bev_mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            query: [B, Q, C]
            mlvl_feats: list of [B, num_cams, C, H, W], one per level.
            reference_points_cam: [num_cams, B, Q, D, 2] normalized [0,1].
            bev_mask: [num_cams, B, Q, D] bool, True = valid projection.
        Returns:
            [B, Q, C]
        """
        batch, num_query, embed_dims = query.shape
        spatial_shapes = [(feat.shape[-2], feat.shape[-1]) for feat in mlvl_feats]

        output_sum = query.new_zeros(batch, num_query, embed_dims)
        weight_sum = query.new_zeros(batch, num_query, 1)

        for cam in range(self.num_cams):
            value = torch.cat(
                [feat[:, cam].flatten(2).transpose(1, 2) for feat in mlvl_feats], dim=1
            )  # [B, S, C]

            ref_points_cam = reference_points_cam[cam]  # [B, Q, D, 2]
            num_points_in_pillar = ref_points_cam.shape[2]
            ref_points_expanded = ref_points_cam[:, :, None, :, :].expand(
                batch, num_query, self.num_levels, num_points_in_pillar, 2
            )

            invalid_mask = ~bev_mask[cam]  # [B, Q, D]
            point_mask = invalid_mask[:, :, None, :].expand(
                batch, num_query, self.num_levels, num_points_in_pillar
            )

            out_cam = self.deform_attn(query, ref_points_expanded, value, spatial_shapes, point_mask=point_mask)
            cam_valid = bev_mask[cam].any(dim=-1).to(query.dtype).unsqueeze(-1)  # [B, Q, 1]

            output_sum = output_sum + out_cam * cam_valid
            weight_sum = weight_sum + cam_valid

        return output_sum / weight_sum.clamp(min=1.0)

In [22]:
# --- Probe: 3 cameras, one BEV query invisible to all of them ---
sca_probe = SpatialCrossAttention(embed_dims=16, num_cams=3, num_levels=1, num_points_in_pillar=3, num_heads=4)
sca_mlvl_feats = [torch.randn(1, 3, 16, 4, 4)]
sca_query = torch.randn(1, 2, 16)
sca_ref_points_cam = torch.rand(3, 1, 2, 3, 2)
sca_bev_mask = torch.ones(3, 1, 2, 3, dtype=torch.bool)
sca_bev_mask[:, :, 0, :] = False  # query 0: no camera sees any of its pillar points

sca_out = sca_probe(sca_query, sca_mlvl_feats, sca_ref_points_cam, sca_bev_mask)
assert sca_out.shape == (1, 2, 16)
assert torch.all(sca_out[:, 0] == 0.0)     # unseen query -> exact zero, not NaN
assert torch.isfinite(sca_out).all()
print("SpatialCrossAttention ok:", sca_out.shape, " unseen-query output:", sca_out[0, 0, :4].tolist())

SpatialCrossAttention ok: torch.Size([1, 2, 16])  unseen-query output: [0.0, 0.0, 0.0, 0.0]


## Temporal Self-Attention: Fusing The Previous BEV Frame

BEVFormer's second novel attention fuses the *current* BEV query state with
the *previous* frame's BEV features (already aligned to the current ego pose
by the warp in the next section) — giving the model access to short-term
motion cues (e.g. velocity) that a single frame cannot provide.

The trick is to treat $[\text{prev\_bev},\ \text{current\_query}]$ as **two
pseudo-levels of the same $H_{\text{bev}}{\times}W_{\text{bev}}$ spatial grid**
and reuse `MultiScaleDeformableAttention` with `num_levels=2`. Because both
"levels" share the same grid layout, a query's own BEV location
$q_{i,j}$ (from the earlier grid-points section) is a *perfectly aligned*
reference point in **both** levels — no projection needed, just a few learned
offset points to look slightly around that same location in each level:

$$\text{TSA}(q) = \text{DeformAttn}\big(z_q,\ r = q_{i,j}\ \text{(shared across both levels)},\ V = [\text{prev\_bev} \Vert \text{current\_query}]\big)$$

At the very first frame of a sequence (no history yet), `prev_bev` is simply
set to the current query itself — the model attends to "the previous frame"
that is, degenerately, itself, which is a graceful fallback rather than a
special-cased branch.

**Contract:** `query:[B,Q,C]`, `prev_bev:[B,Q,C]` or `None`, `bev_h,bev_w` (so
`Q = bev_h*bev_w`) `-> [B,Q,C]`.

In [23]:
class TemporalSelfAttention(nn.Module):
    """Treats [previous BEV, current BEV] as 2 pseudo-levels of the same
    spatial grid and runs deformable attention with each query's own BEV
    grid location as the shared reference point across both levels."""

    def __init__(self, embed_dims: int, num_heads: int = 8, num_points: int = 4) -> None:
        super().__init__()
        self.deform_attn = MultiScaleDeformableAttention(
            embed_dims, num_heads, num_levels=2, num_points=num_points
        )

    def forward(
        self,
        query: torch.Tensor,
        prev_bev: torch.Tensor | None,
        bev_h: int,
        bev_w: int,
    ) -> torch.Tensor:
        batch, num_query, _ = query.shape
        if prev_bev is None:
            prev_bev = query

        value = torch.cat([prev_bev, query], dim=1)  # levels: [prev, current]
        spatial_shapes = [(bev_h, bev_w), (bev_h, bev_w)]

        grid_points = get_bev_grid_points_2d(bev_h, bev_w).to(device=query.device, dtype=query.dtype)  # [Q, 2]
        num_points = self.deform_attn.num_points
        reference_points = grid_points[None, :, None, None, :].expand(batch, num_query, 2, num_points, 2)

        return self.deform_attn(query, reference_points, value, spatial_shapes)

In [24]:
# --- Probe: with and without prev_bev, same output shape either way ---
tsa_probe = TemporalSelfAttention(embed_dims=16, num_heads=4, num_points=2)
tsa_bev_h = tsa_bev_w = 4
tsa_query = torch.randn(2, tsa_bev_h * tsa_bev_w, 16)

tsa_out_no_history = tsa_probe(tsa_query, prev_bev=None, bev_h=tsa_bev_h, bev_w=tsa_bev_w)
tsa_out_with_history = tsa_probe(tsa_query, prev_bev=torch.randn(2, 16, 16), bev_h=tsa_bev_h, bev_w=tsa_bev_w)

assert tsa_out_no_history.shape == tsa_out_with_history.shape == (2, 16, 16)
print("TemporalSelfAttention ok, no-history:", tsa_out_no_history.shape, " with-history:", tsa_out_with_history.shape)

TemporalSelfAttention ok, no-history: torch.Size([2, 16, 16])  with-history: torch.Size([2, 16, 16])


## BEV Positional Encoding: A Learned, Factorized Grid

Both attentions above operate on the *content* of a BEV query, but the
attention modules' linear layers have no other way of knowing *which* cell
$(i,j)$ a query is, information they need to predict sensible reference-point
offsets. A learned, factorized 2D positional encoding supplies this cheaply
— one embedding table per row, one per column, summed:

$$\text{pos}(i,j) = E_{\text{row}}[i] + E_{\text{col}}[j] \in \mathbb{R}^C$$

This costs $O(H_{\text{bev}} + W_{\text{bev}})$ learned parameters instead of
$O(H_{\text{bev}} \cdot W_{\text{bev}})$ for a full unfactorized table, and is
added to the query (not concatenated) before each encoder layer's attentions,
following standard transformer practice.

**Contract:** `() -> [B, H_bev*W_bev, C]` (broadcast to the batch).

In [25]:
class LearnedBEVPositionalEncoding(nn.Module):
    def __init__(self, bev_h: int, bev_w: int, embed_dims: int) -> None:
        super().__init__()
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.row_embed = nn.Embedding(bev_h, embed_dims)
        self.col_embed = nn.Embedding(bev_w, embed_dims)

    def forward(self, batch_size: int) -> torch.Tensor:
        device = self.row_embed.weight.device
        rows = self.row_embed(torch.arange(self.bev_h, device=device))  # [H, C]
        cols = self.col_embed(torch.arange(self.bev_w, device=device))  # [W, C]
        pos = rows[:, None, :] + cols[None, :, :]  # [H, W, C]
        pos = pos.reshape(1, self.bev_h * self.bev_w, -1).expand(batch_size, -1, -1)
        return pos

In [26]:
# --- Probe: distinct cells get distinct codes; same batch index shares one ---
pos_enc_probe = LearnedBEVPositionalEncoding(bev_h=4, bev_w=5, embed_dims=16)
pos_enc_out = pos_enc_probe(batch_size=3)
assert pos_enc_out.shape == (3, 20, 16)
assert not torch.allclose(pos_enc_out[0, 0], pos_enc_out[0, 1])  # different cells differ
torch.testing.assert_close(pos_enc_out[0], pos_enc_out[1])        # same cell, same code across batch
print("LearnedBEVPositionalEncoding ok:", pos_enc_out.shape)

LearnedBEVPositionalEncoding ok: torch.Size([3, 20, 16])


## Warping The Previous BEV Into The Current Ego Frame

`prev_bev` was computed in the *previous* frame's ego-centered coordinate
frame. Before temporal self-attention can treat it as spatially aligned with
the current query grid, it must be resampled into the *current* frame — a
rigid 2D transform (rotate by the ego's yaw change, translate by its
position change), applied with a standard `affine_grid` + `grid_sample`
image warp.

Given ego translation delta $(\Delta x,\Delta y)$ (meters, current minus
previous) and yaw delta $\Delta\theta$, and `pc_range` spans
$s_x = x_{\max}-x_{\min}$, $s_y = y_{\max}-y_{\min}$, the affine matrix
mapping an *output* (current-frame) grid coordinate to the *input*
(previous-frame) sampling coordinate is

$$\Theta = \begin{bmatrix}\cos\Delta\theta & -\sin\Delta\theta & \dfrac{2\Delta x}{s_x} \\[4pt] \sin\Delta\theta & \cos\Delta\theta & \dfrac{2\Delta y}{s_y}\end{bmatrix}$$

(the $2/s$ factor rescales meters into `grid_sample`'s $[-1,1]$ normalized
coordinate convention). $\text{out}(p) = \text{BilinearSample}(\text{prev\_bev},\ \Theta p)$
— cells that warp in from outside the previous grid are zero-padded (the
scene just entered view and has no history yet).

Concretely: if the ego moved $+\Delta x$ forward, static world content
appears to have moved $-\Delta x$ backward in the new ego frame — exactly
what this transform produces, verified below.

**Contract:** `prev_bev:[B,H_{bev}W_{bev},C] -> [B,H_{bev}W_{bev},C]` (shape-preserving).

In [27]:
def warp_prev_bev(
    prev_bev: torch.Tensor,
    bev_h: int,
    bev_w: int,
    delta_translation_bev: torch.Tensor,
    delta_yaw: torch.Tensor,
    pc_range: tuple[float, float, float, float, float, float],
) -> torch.Tensor:
    """
    Args:
        prev_bev: [B, bev_h*bev_w, C], row-major (row=y, col=x).
        delta_translation_bev: [B, 2] ego translation (x, y) in meters,
            current frame minus previous frame.
        delta_yaw: [B] ego yaw change in radians, current minus previous.
        pc_range: [xmin, ymin, zmin, xmax, ymax, zmax].
    Returns:
        [B, bev_h*bev_w, C]: prev_bev resampled into the current ego frame.
    """
    batch, _, embed_dims = prev_bev.shape
    feature_map = prev_bev.reshape(batch, bev_h, bev_w, embed_dims).permute(0, 3, 1, 2)  # [B, C, H, W]

    span_x = pc_range[3] - pc_range[0]
    span_y = pc_range[4] - pc_range[1]
    tx = 2.0 * delta_translation_bev[:, 0] / span_x
    ty = 2.0 * delta_translation_bev[:, 1] / span_y

    cos = torch.cos(delta_yaw)
    sin = torch.sin(delta_yaw)
    theta = torch.zeros(batch, 2, 3, dtype=prev_bev.dtype, device=prev_bev.device)
    theta[:, 0, 0] = cos
    theta[:, 0, 1] = -sin
    theta[:, 0, 2] = tx
    theta[:, 1, 0] = sin
    theta[:, 1, 1] = cos
    theta[:, 1, 2] = ty

    grid = F.affine_grid(theta, feature_map.shape, align_corners=False)
    warped = F.grid_sample(feature_map, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    return warped.permute(0, 2, 3, 1).reshape(batch, bev_h * bev_w, embed_dims)

In [28]:
# --- Probe: zero delta is identity; a +1-cell ego motion shifts content back by 1 cell ---
warp_pc_range = (-10.0, -10.0, -2.0, 10.0, 10.0, 2.0)  # 20m span, 5 cells -> 4m/cell
warp_bev_h = warp_bev_w = 5

warp_random_bev = torch.randn(1, warp_bev_h * warp_bev_w, 4)
warp_identity_out = warp_prev_bev(warp_random_bev, warp_bev_h, warp_bev_w, torch.zeros(1, 2), torch.zeros(1), warp_pc_range)
torch.testing.assert_close(warp_identity_out, warp_random_bev, atol=1e-4, rtol=1e-4)

warp_hot_pixel = torch.zeros(1, warp_bev_h, warp_bev_w, 1)
warp_hot_pixel[0, 2, 2, 0] = 1.0  # a single "landmark" at the center cell
warp_hot_pixel = warp_hot_pixel.reshape(1, warp_bev_h * warp_bev_w, 1)

warp_cell_size_x = (warp_pc_range[3] - warp_pc_range[0]) / warp_bev_w  # 4.0 m
warp_shifted = warp_prev_bev(warp_hot_pixel, warp_bev_h, warp_bev_w, torch.tensor([[warp_cell_size_x, 0.0]]), torch.zeros(1), warp_pc_range)
warp_shifted_grid = warp_shifted.reshape(warp_bev_h, warp_bev_w)
peak_row, peak_col = (warp_shifted_grid == warp_shifted_grid.max()).nonzero()[0].tolist()
assert (peak_row, peak_col) == (2, 1)  # landmark now appears one cell to the -x side
print("warp_prev_bev ok: zero-delta identity verified; +1-cell ego motion moved landmark from col 2 to col 1")

warp_prev_bev ok: zero-delta identity verified; +1-cell ego motion moved landmark from col 2 to col 1


## BEVFormerLayer And BEVFormerEncoder: Putting The BEV Encoder Together

One encoder layer chains the two attentions with post-norm residual
connections and a feed-forward network (standard Transformer layer
structure, following Vaswani et al.):

$$z \leftarrow \operatorname{LN}\big(z + \text{TSA}(z + \text{pos},\ \text{prev\_bev})\big)$$
$$z \leftarrow \operatorname{LN}\big(z + \text{SCA}(z + \text{pos},\ \text{mlvl\_feats}, \dots)\big)$$
$$z \leftarrow \operatorname{LN}\big(z + \operatorname{FFN}(z)\big), \qquad \operatorname{FFN}(z)=W_2\,\operatorname{ReLU}(W_1 z)$$

The encoder wraps $N_{\text{layers}}$ of these, feeding each layer's output
$z$ as the next layer's input (**iterative refinement** — the BEV
representation is built up progressively, not produced in one shot) while
reusing the *same* warped `prev_bev` and camera-projected pillar reference
points at every layer (both are computed once per frame, not per layer, for
efficiency):

$$\text{bev\_embed} = \text{Layer}_{N}\big(\cdots \text{Layer}_1(\text{learned BEV query embedding})\cdots\big)$$

The learned BEV query embedding (an `nn.Embedding(H_{bev} \cdot W_{bev}, C)`)
is the encoder's *only* input besides the image features and previous BEV —
there is no "zero" or "random" initial BEV state; the network learns what an
empty/uninformed BEV grid should look like before any evidence is fused in.

In [29]:
class BEVFormerLayer(nn.Module):
    def __init__(
        self,
        embed_dims: int,
        num_heads: int,
        num_levels: int,
        num_points_in_pillar: int,
        num_points_temporal: int,
        num_cams: int,
        feedforward_dims: int,
    ) -> None:
        super().__init__()
        self.temporal_self_attn = TemporalSelfAttention(embed_dims, num_heads, num_points_temporal)
        self.spatial_cross_attn = SpatialCrossAttention(
            embed_dims, num_cams, num_levels, num_points_in_pillar, num_heads
        )
        self.norm1 = nn.LayerNorm(embed_dims)
        self.norm2 = nn.LayerNorm(embed_dims)
        self.norm3 = nn.LayerNorm(embed_dims)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dims, feedforward_dims),
            nn.ReLU(inplace=True),
            nn.Linear(feedforward_dims, embed_dims),
        )

    def forward(
        self,
        query: torch.Tensor,
        mlvl_feats: list[torch.Tensor],
        reference_points_cam: torch.Tensor,
        bev_mask: torch.Tensor,
        prev_bev: torch.Tensor | None,
        bev_pos: torch.Tensor,
        bev_h: int,
        bev_w: int,
    ) -> torch.Tensor:
        tsa_out = self.temporal_self_attn(query + bev_pos, prev_bev, bev_h, bev_w)
        query = self.norm1(query + tsa_out)

        sca_out = self.spatial_cross_attn(query + bev_pos, mlvl_feats, reference_points_cam, bev_mask)
        query = self.norm2(query + sca_out)

        ffn_out = self.ffn(query)
        query = self.norm3(query + ffn_out)
        return query

In [30]:
class BEVFormerEncoder(nn.Module):
    def __init__(
        self,
        num_layers: int,
        bev_h: int,
        bev_w: int,
        embed_dims: int,
        pc_range: tuple[float, float, float, float, float, float],
        num_cams: int,
        num_heads: int = 8,
        num_levels: int = 4,
        num_points_in_pillar: int = 4,
        num_points_temporal: int = 4,
        feedforward_dims: int = 512,
    ) -> None:
        super().__init__()
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.pc_range = pc_range
        self.num_points_in_pillar = num_points_in_pillar

        self.bev_embedding = nn.Embedding(bev_h * bev_w, embed_dims)
        self.positional_encoding = LearnedBEVPositionalEncoding(bev_h, bev_w, embed_dims)
        self.layers = nn.ModuleList(
            [
                BEVFormerLayer(
                    embed_dims,
                    num_heads,
                    num_levels,
                    num_points_in_pillar,
                    num_points_temporal,
                    num_cams,
                    feedforward_dims,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        mlvl_feats: list[torch.Tensor],
        img_metas: list[dict],
        prev_bev: torch.Tensor | None = None,
        delta_translation_bev: torch.Tensor | None = None,
        delta_yaw: torch.Tensor | None = None,
    ) -> torch.Tensor:
        batch = mlvl_feats[0].shape[0]
        device = mlvl_feats[0].device

        query = self.bev_embedding.weight.unsqueeze(0).expand(batch, -1, -1).to(device)
        bev_pos = self.positional_encoding(batch).to(device)

        if prev_bev is not None:
            prev_bev = warp_prev_bev(
                prev_bev, self.bev_h, self.bev_w, delta_translation_bev, delta_yaw, self.pc_range
            )

        reference_points_3d = get_pillar_reference_points_3d(
            self.bev_h, self.bev_w, self.pc_range, self.num_points_in_pillar
        ).to(device)
        reference_points_cam, bev_mask = project_pillar_points_to_cameras(
            reference_points_3d, self.pc_range, img_metas
        )
        reference_points_cam = reference_points_cam.to(device)
        bev_mask = bev_mask.to(device)

        for layer in self.layers:
            query = layer(
                query, mlvl_feats, reference_points_cam, bev_mask, prev_bev, bev_pos, self.bev_h, self.bev_w
            )
        return query

In [31]:
# --- Probe: full single-frame encoder forward, with and without prev_bev ---
encoder_probe = BEVFormerEncoder(
    num_layers=2, bev_h=4, bev_w=4, embed_dims=16, pc_range=warp_pc_range,
    num_cams=2, num_heads=4, num_levels=1, num_points_in_pillar=3,
    num_points_temporal=2, feedforward_dims=32,
)
encoder_mlvl_feats = [torch.randn(1, 2, 16, 4, 4)]
encoder_img_metas = [{"lidar2img": [identity_lidar2img(), identity_lidar2img()], "image_size": (100, 100)}]

bev_embed_frame0 = encoder_probe(encoder_mlvl_feats, encoder_img_metas)
assert bev_embed_frame0.shape == (1, 16, 16)  # [B, H_bev*W_bev, C]

bev_embed_frame1 = encoder_probe(
    encoder_mlvl_feats, encoder_img_metas,
    prev_bev=bev_embed_frame0, delta_translation_bev=torch.zeros(1, 2), delta_yaw=torch.zeros(1),
)
assert bev_embed_frame1.shape == bev_embed_frame0.shape
print("BEVFormerEncoder ok: bev_embed", bev_embed_frame0.shape, "(frame 0, no history) and", bev_embed_frame1.shape, "(frame 1, with history)")

BEVFormerEncoder ok: bev_embed torch.Size([1, 16, 16]) (frame 0, no history) and torch.Size([1, 16, 16]) (frame 1, with history)


## BEVFormerHead: From Decoder Hidden State To A 3D Box

Each object query's final decoder hidden state $z_q$ is turned into a class
score and a box by two small per-decoder-layer MLPs, `cls_branches[l]` and
`reg_branches[l]` (independent weights per layer — later layers see more
refined queries and can specialize).

**Classification** is a direct $K$-way logit vector — sigmoid'd,
multi-label style (each class scored independently, not softmax'd), the
standard DETR / focal-loss convention:

$$\text{cls\_score}_q = \operatorname{MLP}_{\text{cls}}(z_q) \in \mathbb{R}^K$$

**Box regression** predicts a *residual* on top of the query's own reference
point $r_q = (r_x,r_y,r_z) \in [0,1]^3$ (a `Linear(C,3)+sigmoid` applied to
the query embedding, refined per decoder layer — see the next section), not
an absolute box, so that a query only has to learn "nudge me a little" rather
than re-derive its own rough location every layer:

$$\text{reg}_q = \operatorname{MLP}_{\text{reg}}(z_q) \in \mathbb{R}^{10}$$
$$\hat x = \operatorname{sigmoid}\big(\text{reg}_{q,0} + \operatorname{logit}(r_x)\big), \quad
  \hat y = \operatorname{sigmoid}\big(\text{reg}_{q,1} + \operatorname{logit}(r_y)\big)$$
$$\hat z = \operatorname{sigmoid}\big(\text{reg}_{q,4} + \operatorname{logit}(r_z)\big)$$
$$(x,y,z) = \text{denormalize}\big((\hat x,\hat y,\hat z),\ \texttt{pc\_range}\big)$$

The other 7 regression channels are used *as-is* (no reference-point
composition, since width/length/height/yaw/velocity have no natural $[0,1]$
"reference"), giving the final 10D encoded box
$[x, y, \log w, \log l, z, \log h, \sin\psi, \cos\psi, v_x, v_y]$
— the same box parameterization defined in the loss-utilities section below.

Both branches' final layers are bias-initialized deliberately: classification
to $-\ln\frac{1-\pi}{\pi}$ with $\pi{=}0.01$ (focal-loss convention — most
queries should start confidently predicting "no object", since true positives
are rare among hundreds of queries), regression to zero (start as a no-op
residual on top of the reference point).

In [32]:
class MLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        num_layers: int,
        *,
        use_layernorm: bool = False,
    ) -> None:
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(in_dim, hidden_dim))
            if use_layernorm:
                layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.ReLU(inplace=True))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [33]:
class BEVFormerHead(nn.Module):
    def __init__(
        self,
        embed_dims: int = 256,
        num_classes: int = 10,
        box_dim: int = 10,
        num_decoder_layers: int = 6,
        pc_range: tuple[float, float, float, float, float, float] = (-51.2, -51.2, -5.0, 51.2, 51.2, 3.0),
    ) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.box_dim = box_dim
        self.num_decoder_layers = num_decoder_layers
        self.pc_range = pc_range
        self.reference_points = nn.Linear(embed_dims, 3)
        self.cls_branches = nn.ModuleList(
            [MLP(embed_dims, embed_dims, num_classes, num_layers=3, use_layernorm=True) for _ in range(num_decoder_layers)]
        )
        self.reg_branches = nn.ModuleList(
            [MLP(embed_dims, embed_dims, box_dim, num_layers=3) for _ in range(num_decoder_layers)]
        )
        self._init_weights()

    def _init_weights(self) -> None:
        prior_prob = 0.01
        cls_bias = -math.log((1 - prior_prob) / prior_prob)

        nn.init.xavier_uniform_(self.reference_points.weight)
        nn.init.constant_(self.reference_points.bias, 0.0)

        for cls_branch in self.cls_branches:
            final_linear = cls_branch.net[-1]
            nn.init.constant_(final_linear.bias, cls_bias)

        for reg_branch in self.reg_branches:
            final_linear = reg_branch.net[-1]
            nn.init.constant_(final_linear.bias, 0.0)

    def _encode_box_predictions(self, reg_output: torch.Tensor, reference_points: torch.Tensor) -> torch.Tensor:
        if self.box_dim != 10:
            raise ValueError(f"Expected official-style box_dim=10, got {self.box_dim}")

        ref_logits = inverse_sigmoid(reference_points)
        center_xy_norm = (reg_output[..., 0:2] + ref_logits[..., 0:2]).sigmoid()
        center_z_norm = (reg_output[..., 4:5] + ref_logits[..., 2:3]).sigmoid()

        center_xyz = denormalize_reference_points(
            torch.cat([center_xy_norm, center_z_norm], dim=-1),
            self.pc_range,
        )

        encoded = reg_output.clone()
        encoded[..., 0:1] = center_xyz[..., 0:1]
        encoded[..., 1:2] = center_xyz[..., 1:2]
        encoded[..., 4:5] = center_xyz[..., 2:3]
        return encoded

    def init_reference_points(self, query_pos: torch.Tensor) -> torch.Tensor:
        return self.reference_points(query_pos).sigmoid()

    def predict_reference_points(self, layer_idx: int, layer_q: torch.Tensor) -> torch.Tensor:
        return self.reference_points(layer_q).sigmoid()

    def regress_boxes(self, layer_idx: int, layer_hs: torch.Tensor) -> torch.Tensor:
        return self.reg_branches[layer_idx](layer_hs)

    def classify(self, layer_idx: int, layer_hs: torch.Tensor) -> torch.Tensor:
        return self.cls_branches[layer_idx](layer_hs)

    def forward_single(
        self,
        layer_idx: int,
        layer_hs: torch.Tensor,
        reference_points: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        reg_output = self.regress_boxes(layer_idx, layer_hs)
        cls_score = self.classify(layer_idx, layer_hs)
        bbox_pred = self._encode_box_predictions(reg_output, reference_points)
        return cls_score, bbox_pred

    def forward(self, hs: torch.Tensor, inter_references: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        cls_scores = []
        bbox_preds = []
        for layer_idx, layer_hs in enumerate(hs):
            reference_points = inter_references[layer_idx]
            cls_score, bbox_pred = self.forward_single(layer_idx, layer_hs, reference_points)
            cls_scores.append(cls_score)
            bbox_preds.append(bbox_pred)
        return torch.stack(cls_scores), torch.stack(bbox_preds)

In [34]:
# --- Probe: reference points in range, per-layer cls/box shapes ---
head_probe = BEVFormerHead(embed_dims=16, num_classes=3, box_dim=10, num_decoder_layers=2, pc_range=warp_pc_range)

head_query_pos = torch.randn(1, 6, 16)
head_ref_points = head_probe.init_reference_points(head_query_pos)
assert head_ref_points.shape == (1, 6, 3)
assert torch.all((head_ref_points >= 0) & (head_ref_points <= 1))

head_hs = torch.randn(2, 1, 6, 16)                    # [num_layers, B, Q, C]
head_inter_refs = torch.rand(2, 1, 6, 3)
head_cls_scores, head_bbox_preds = head_probe(head_hs, head_inter_refs)
assert head_cls_scores.shape == (2, 1, 6, 3)   # [L, B, Q, K]
assert head_bbox_preds.shape == (2, 1, 6, 10)  # [L, B, Q, 10]
print("BEVFormerHead ok: reference_points", head_ref_points.shape, " cls_scores", head_cls_scores.shape, " bbox_preds", head_bbox_preds.shape)

BEVFormerHead ok: reference_points torch.Size([1, 6, 3])  cls_scores torch.Size([2, 1, 6, 3])  bbox_preds torch.Size([2, 1, 6, 10])


## BEVFormerDecoder: Object Queries Reading From The BEV Grid

$Q$ learned object queries (`query_embed`) — one per potential detection,
independent of the BEV grid's own $H_{\text{bev}}{\cdot}W_{\text{bev}}$
queries — are refined by $N_{\text{dec}}$ decoder layers, each with:

**Self-attention** among the $Q$ queries (ordinary scaled dot-product,
`nn.MultiheadAttention`) — lets queries "negotiate" so two queries don't
converge on the same object:

$$z \leftarrow \operatorname{LN}\big(z + \operatorname{SelfAttn}(z+\text{pos},\ z+\text{pos},\ z)\big)$$

**Deformable cross-attention** against the *single* flattened BEV feature map
(`bev_embed`, `num_levels=1`) — this reuses the exact same
`MultiScaleDeformableAttention` core from the encoder, just with the object
query's own predicted 2D reference point $(\hat x,\hat y)$ (from the head, see
above) instead of a fixed BEV grid cell:

$$z \leftarrow \operatorname{LN}\big(z + \text{DeformAttn}(z+\text{pos},\ r=(\hat x,\hat y),\ V{=}\text{bev\_embed})\big)$$

then an FFN + residual, identical in form to the encoder layer. Reference
points are **re-predicted from the current query state before every layer**
(via `reference_point_predictor`, the head's `predict_reference_points`) —
each layer sees a fresh, progressively refined guess at "where is this
object", not a single guess fixed at layer 0.

**Contract:** `bev_embed:[B,H_{bev}W_{bev},C] -> (hidden_states:[N_{dec},B,Q,C],
init_reference:[B,Q,3], inter_references:[N_{dec},B,Q,3])`.

In [35]:
class BEVFormerDecoderLayer(nn.Module):
    def __init__(
        self,
        embed_dims: int = 256,
        num_heads: int = 8,
        num_points: int = 4,
        ffn_channels: int = 512,
    ) -> None:
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dims, num_heads, batch_first=True)
        self.cross_attn = MultiScaleDeformableAttention(embed_dims, num_heads, num_levels=1, num_points=num_points)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dims, ffn_channels),
            nn.ReLU(inplace=True),
            nn.Linear(ffn_channels, embed_dims),
        )
        self.norm1 = nn.LayerNorm(embed_dims)
        self.norm2 = nn.LayerNorm(embed_dims)
        self.norm3 = nn.LayerNorm(embed_dims)

    def forward(
        self,
        query: torch.Tensor,
        query_pos: torch.Tensor,
        reference_points_xy: torch.Tensor,
        value: torch.Tensor,
        spatial_shapes: list[tuple[int, int]],
    ) -> torch.Tensor:
        q = query + query_pos
        self_attended, _ = self.self_attn(q, q, query)
        query = self.norm1(query + self_attended)

        num_points = self.cross_attn.num_points
        reference_points = reference_points_xy[:, :, None, None, :].expand(
            reference_points_xy.shape[0], reference_points_xy.shape[1], 1, num_points, 2
        )
        cross = self.cross_attn(query + query_pos, reference_points, value, spatial_shapes)
        query = self.norm2(query + cross)

        ffn_out = self.ffn(query)
        return self.norm3(query + ffn_out)

In [36]:
class BEVFormerDecoder(nn.Module):
    def __init__(
        self,
        embed_dims: int = 256,
        num_queries: int = 900,
        num_layers: int = 6,
        num_heads: int = 8,
        num_points: int = 4,
        ffn_channels: int = 512,
    ) -> None:
        super().__init__()
        self.embed_dims = embed_dims
        self.num_queries = num_queries
        self.query_embed = nn.Embedding(num_queries, embed_dims)
        self.query_pos = nn.Embedding(num_queries, embed_dims)
        self.layers = nn.ModuleList(
            [BEVFormerDecoderLayer(embed_dims, num_heads, num_points, ffn_channels) for _ in range(num_layers)]
        )

    def init_decoder_state(self, batch_size: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
        query = self.query_embed.weight.unsqueeze(0).expand(batch_size, -1, -1).to(device)
        query_pos = self.query_pos.weight.unsqueeze(0).expand(batch_size, -1, -1).to(device)
        return query, query_pos

    def forward(
        self,
        bev_embed: torch.Tensor,
        bev_h: int,
        bev_w: int,
        reference_point_predictor: Callable[[int, torch.Tensor], torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch = bev_embed.shape[0]
        query, query_pos = self.init_decoder_state(batch, bev_embed.device)
        spatial_shapes = [(bev_h, bev_w)]

        init_reference = None
        intermediate_states = []
        intermediate_refs = []
        hidden = query
        for layer_idx, layer in enumerate(self.layers):
            reference_points = reference_point_predictor(layer_idx, hidden)
            if init_reference is None:
                init_reference = reference_points
            intermediate_refs.append(reference_points)
            hidden = layer(hidden, query_pos, reference_points[..., :2], bev_embed, spatial_shapes)
            intermediate_states.append(hidden)

        return torch.stack(intermediate_states), init_reference, torch.stack(intermediate_refs)

In [37]:
# --- Probe: full decoder forward against a synthetic bev_embed ---
decoder_probe = BEVFormerDecoder(embed_dims=16, num_queries=6, num_layers=2, num_heads=4, num_points=2, ffn_channels=32)
decoder_bev_embed = torch.randn(1, 16, 16)  # [B, H_bev*W_bev=16, C]

decoder_hidden_states, decoder_init_ref, decoder_inter_refs = decoder_probe(
    decoder_bev_embed, bev_h=4, bev_w=4, reference_point_predictor=head_probe.predict_reference_points,
)
assert decoder_hidden_states.shape == (2, 1, 6, 16)  # [N_dec, B, Q, C]
assert decoder_init_ref.shape == (1, 6, 3)
assert decoder_inter_refs.shape == (2, 1, 6, 3)
print("BEVFormerDecoder ok: hidden_states", decoder_hidden_states.shape, " inter_references", decoder_inter_refs.shape)

BEVFormerDecoder ok: hidden_states torch.Size([2, 1, 6, 16])  inter_references torch.Size([2, 1, 6, 3])


## Box Parameterization: Semantic <-> Encoded

Ground-truth and predicted boxes are converted between two representations.
**Semantic** (9D, human-readable): $[x,y,z,w,l,h,\psi,v_x,v_y]$ — center,
size, yaw angle, velocity. **Encoded** (10D, what the network actually
predicts/is supervised on): sizes in log-space (so a small regression error
scales multiplicatively, appropriate for physical sizes spanning meters to
tens of meters) and yaw as $(\sin\psi,\cos\psi)$ instead of the angle itself
(avoids the discontinuity at $\pm\pi$ that a raw angle regression would hit):

$$\text{encode}(x,y,z,w,l,h,\psi,v_x,v_y) = \big[x,\ y,\ \ln w,\ \ln l,\ z,\ \ln h,\ \sin\psi,\ \cos\psi,\ v_x,\ v_y\big]$$

Decoding inverts this exactly, recovering yaw via $\psi=\operatorname{atan2}(\sin\psi,\cos\psi)$
— guaranteed to be well-defined and wrapped to $(-\pi,\pi]$ regardless of the
network's raw $(\sin,\cos)$ outputs (they need not even lie exactly on the
unit circle for `atan2` to still return a sensible angle).

A related helper, `wrapped_yaw_difference`, computes an angular difference
correctly across the $\pm\pi$ wrap-around — naively subtracting two angles
near $+\pi$ and $-\pi$ gives $\approx 2\pi$ instead of the true near-zero
difference:

$$\Delta\psi = \operatorname{atan2}\big(\sin(\psi_1-\psi_2),\ \cos(\psi_1-\psi_2)\big)$$

In [38]:
def encode_bbox_targets(boxes: torch.Tensor, pc_range=None) -> torch.Tensor:
    """Encode semantic 9D boxes into the 10D training target.

    Input: `[x, y, z, w, l, h, yaw, vx, vy]`
    Output: `[x, y, log(w), log(l), z, log(h), sin(yaw), cos(yaw), vx, vy]`
    """
    if boxes.numel() == 0:
        return boxes.new_zeros((0, 10))
    if boxes.shape[-1] != 9:
        raise ValueError(f"Expected 9D semantic boxes, got {boxes.shape[-1]}")

    x = boxes[..., 0:1]
    y = boxes[..., 1:2]
    z = boxes[..., 2:3]
    w = boxes[..., 3:4].clamp(min=1e-5).log()
    l = boxes[..., 4:5].clamp(min=1e-5).log()
    h = boxes[..., 5:6].clamp(min=1e-5).log()
    yaw = boxes[..., 6:7]
    vx = boxes[..., 7:8]
    vy = boxes[..., 8:9]
    return torch.cat([x, y, w, l, z, h, yaw.sin(), yaw.cos(), vx, vy], dim=-1)

In [39]:
def wrapped_yaw_difference(pred_yaw: torch.Tensor, target_yaw: torch.Tensor) -> torch.Tensor:
    diff = pred_yaw - target_yaw
    return torch.atan2(torch.sin(diff), torch.cos(diff))


def decode_bbox_predictions(box_preds: torch.Tensor, pc_range=None) -> torch.Tensor:
    """Decode predictions into semantic `[x, y, z, w, l, h, yaw, vx, vy]`."""
    if box_preds.shape[-1] in (7, 9):
        return box_preds
    if box_preds.shape[-1] != 10:
        raise ValueError(f"Expected bbox dim 7, 9, or 10, got {box_preds.shape[-1]}")

    x = box_preds[..., 0:1]
    y = box_preds[..., 1:2]
    z = box_preds[..., 4:5]
    w = box_preds[..., 2:3].exp()
    l = box_preds[..., 3:4].exp()
    h = box_preds[..., 5:6].exp()
    yaw = torch.atan2(box_preds[..., 6:7], box_preds[..., 7:8])
    vx = box_preds[..., 8:9]
    vy = box_preds[..., 9:10]
    return torch.cat([x, y, z, w, l, h, yaw, vx, vy], dim=-1)

In [40]:
# --- Probe: round trip, and the wrap-around case near +-pi ---
loss_util_box = torch.tensor([[1.0, 2.0, 0.5, 2.0, 4.5, 1.6, 0.3, 1.0, -0.5]])
loss_util_encoded = encode_bbox_targets(loss_util_box)
loss_util_decoded = decode_bbox_predictions(loss_util_encoded)
torch.testing.assert_close(loss_util_decoded, loss_util_box, atol=1e-4, rtol=1e-4)

loss_util_near_pi = torch.tensor([math.pi - 0.1])
loss_util_near_minus_pi = torch.tensor([-math.pi + 0.1])
loss_util_true_diff = wrapped_yaw_difference(loss_util_near_pi, loss_util_near_minus_pi)
assert abs(loss_util_true_diff.item()) < 0.3   # true gap is ~0.2 rad, not ~2*pi
print("encode/decode round trip ok; wrapped yaw diff near +-pi:", loss_util_true_diff.item(), "rad (naive subtraction would give ~", (loss_util_near_pi - loss_util_near_minus_pi).item(), ")")

encode/decode round trip ok; wrapped yaw diff near +-pi: -0.2000001072883606 rad (naive subtraction would give ~ 6.083185195922852 )


## HungarianMatcher3D: Assigning Predictions To Ground Truth

DETR-style training needs a one-to-one assignment between $Q$ predictions and
$G$ ground-truth boxes before a loss can be computed (there's no fixed
"anchor" telling query 7 to always predict the 3rd car). The assignment
minimizing total cost is found by the Hungarian algorithm (`scipy`'s
`linear_sum_assignment`) over a $Q \times G$ cost matrix combining a
classification term and a box term:

**Classification cost** — a *cost* version of focal loss (lower is better,
so it's the *negative* of what would normally be maximized), evaluated only
at each ground truth's true class $c_g$:

$$\text{cost}^{\text{cls}}_{q,g} = \alpha\,(1-p_{q,c_g})^\gamma\big(-\ln p_{q,c_g}\big) \;-\; (1-\alpha)\,p_{q,c_g}^\gamma\big(-\ln(1-p_{q,c_g})\big), \qquad p_{q,k} = \operatorname{sigmoid}(\text{logit}_{q,k})$$

**Box cost** — L1 distance in encoded-box space, excluding the 2 velocity
channels (matching official DETR3D/BEVFormer, since velocity is comparatively
noisy and shouldn't drive the discrete assignment):

$$\text{cost}^{\text{box}}_{q,g} = \big\Vert \text{box\_preds}_q^{(0:8)} - \text{encode}(\text{gt}_g)^{(0:8)} \big\Vert_1$$

$$\text{cost}_{q,g} = \lambda_{\text{cls}}\cdot\text{cost}^{\text{cls}}_{q,g} + \lambda_{\text{box}}\cdot\text{cost}^{\text{box}}_{q,g}$$

The assignment runs under `@torch.no_grad()` — it's a discrete combinatorial
choice, not a differentiable operation; gradients flow later through the
*loss* computed at the chosen assignment, not through the assignment itself.
A sample with zero ground-truth boxes gets an empty assignment (every
prediction is background) without needing scipy at all.

In [41]:
def _linear_sum_assignment(cost: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    from scipy.optimize import linear_sum_assignment

    row_ind, col_ind = linear_sum_assignment(cost.detach().cpu().numpy())
    device = cost.device
    return (
        torch.as_tensor(row_ind, dtype=torch.long, device=device),
        torch.as_tensor(col_ind, dtype=torch.long, device=device),
    )


def _box_l1_cost_matrix(pred_boxes: torch.Tensor, gt_boxes: torch.Tensor) -> torch.Tensor:
    pred = pred_boxes[:, None, :].expand(-1, gt_boxes.shape[0], -1)
    gt = gt_boxes[None, :, :].expand(pred_boxes.shape[0], -1, -1)
    return (pred - gt).abs().sum(dim=-1)


def _focal_class_cost(
    cls_logits: torch.Tensor,
    gt_labels: torch.Tensor,
    alpha: float,
    gamma: float,
) -> torch.Tensor:
    probs = cls_logits.sigmoid().clamp(min=1e-8, max=1 - 1e-8)
    neg_cost = -(1 - probs).log() * (1 - alpha) * probs.pow(gamma)
    pos_cost = -(probs).log() * alpha * (1 - probs).pow(gamma)
    return pos_cost[:, gt_labels] - neg_cost[:, gt_labels]

In [42]:
class HungarianMatcher3D:
    def __init__(
        self,
        num_classes: int,
        pc_range=None,
        cls_weight: float = 2.0,
        bbox_weight: float = 0.25,
        alpha: float = 0.25,
        gamma: float = 2.0,
    ) -> None:
        self.num_classes = num_classes
        self.pc_range = pc_range
        self.cls_weight = cls_weight
        self.bbox_weight = bbox_weight
        self.alpha = alpha
        self.gamma = gamma

    @torch.no_grad()
    def __call__(
        self,
        cls_logits: torch.Tensor,
        box_preds: torch.Tensor,
        gt_boxes: list[torch.Tensor],
        gt_labels: list[torch.Tensor],
    ) -> list[tuple[torch.Tensor, torch.Tensor]]:
        cls_logits = cls_logits.float()
        box_preds = box_preds.float()
        assignments = []
        for batch_idx in range(cls_logits.shape[0]):
            if gt_boxes[batch_idx].numel() == 0:
                empty = torch.empty(0, dtype=torch.long, device=cls_logits.device)
                assignments.append((empty, empty))
                continue

            encoded_gt = encode_bbox_targets(gt_boxes[batch_idx].float(), self.pc_range).to(box_preds.dtype)
            cls_cost = _focal_class_cost(
                cls_logits[batch_idx], gt_labels[batch_idx], alpha=self.alpha, gamma=self.gamma
            )
            bbox_cost = _box_l1_cost_matrix(box_preds[batch_idx, :, :8], encoded_gt[:, :8])
            total_cost = self.cls_weight * cls_cost + self.bbox_weight * bbox_cost
            pred_ids, gt_ids = _linear_sum_assignment(total_cost)
            assignments.append((pred_ids, gt_ids))
        return assignments

In [43]:
# --- Probe: one near-perfect prediction should out-compete one wildly-off prediction ---
matcher_probe = HungarianMatcher3D(num_classes=3, pc_range=warp_pc_range)

matcher_gt_box = torch.tensor([[1.0, 2.0, 0.0, 2.0, 4.0, 1.5, 0.1, 0.0, 0.0]])
matcher_gt_label = torch.tensor([1])
matcher_cls_logits = torch.tensor([[[-5.0, 5.0, -5.0], [5.0, -5.0, -5.0]]])  # query0 confidently class1, query1 confidently class0
matcher_good_box = encode_bbox_targets(matcher_gt_box)
matcher_bad_box = torch.zeros_like(matcher_good_box) + 100.0
matcher_box_preds = torch.stack([matcher_good_box[0], matcher_bad_box[0]]).unsqueeze(0)

matcher_assignments = matcher_probe(matcher_cls_logits, matcher_box_preds, [matcher_gt_box], [matcher_gt_label])
matcher_pred_ids, matcher_gt_ids = matcher_assignments[0]
assert matcher_pred_ids.tolist() == [0] and matcher_gt_ids.tolist() == [0]
print("HungarianMatcher3D ok: matched prediction 0 (near-perfect) to ground truth 0, ignored prediction 1 (way off)")

HungarianMatcher3D ok: matched prediction 0 (near-perfect) to ground truth 0, ignored prediction 1 (way off)


## BEVFormerLoss: Focal Classification + Weighted L1, With Auxiliary Supervision

Given the matcher's assignment, every query is labeled either its matched
ground-truth class or "background" (class index $K$, encoded as an all-zero
one-hot row rather than an explicit extra class column — sigmoid focal loss
naturally handles "no positive class" this way).

**Classification** — dense sigmoid focal loss (Lin et al., *RetinaNet*) over
all $Q$ queries, not just matched ones (unmatched queries are supervised
*to be* background):

$$p_t = p\cdot y + (1-p)\cdot(1-y), \qquad w = \big(\alpha y + (1-\alpha)(1-y)\big)(1-p_t)^\gamma$$
$$\mathcal L_{\text{cls}} = \frac{1}{\max(1,\,n_{\text{pos}})}\sum_{q,k} w_{q,k}\cdot \operatorname{BCE}(\text{logit}_{q,k},\, y_{q,k})$$

**Box regression** — weighted L1, computed **only** at matched (query, gt)
pairs (unmatched queries contribute zero box loss — there is nothing to
regress toward):

$$\mathcal L_{\text{box}} = \frac{1}{\max(1,\,n_{\text{pos}})} \sum_{q \in \text{matched}} \big\Vert c \odot (\text{box\_preds}_q - \text{encode}(\text{gt}_{\text{match}(q)})) \big\Vert_1$$

where $c$ is a fixed per-channel `code_weights` vector (down-weighting the
2 velocity channels to $0.2$, matching official configs — velocity is a
smaller, noisier quantity that would otherwise dominate an unweighted L1 sum
next to meter-scale position/size errors).

**Auxiliary losses**: the same two losses are computed independently at
*every* decoder layer's predictions (not just the final one) and all summed
into the total training objective — standard DETR-family practice that gives
earlier layers a direct gradient signal instead of relying entirely on
backpropagation through later layers.

In [44]:
class BEVFormerLoss:
    def __init__(
        self,
        num_classes: int,
        pc_range: tuple[float, float, float, float, float, float] = (-51.2, -51.2, -5.0, 51.2, 51.2, 3.0),
        code_weights: tuple[float, ...] = (1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.2, 0.2),
        matcher_cls_weight: float = 2.0,
        matcher_bbox_weight: float = 0.25,
        loss_cls_weight: float = 2.0,
        loss_bbox_weight: float = 0.25,
        use_auxiliary_losses: bool = True,
        alpha: float = 0.25,
        gamma: float = 2.0,
        bg_cls_weight: float = 0.0,
    ) -> None:
        self.num_classes = num_classes
        self.pc_range = pc_range
        self.loss_cls_weight = loss_cls_weight
        self.loss_bbox_weight = loss_bbox_weight
        self.use_auxiliary_losses = use_auxiliary_losses
        self.code_weights = torch.tensor(code_weights, dtype=torch.float32)
        self.alpha = alpha
        self.gamma = gamma
        self.bg_cls_weight = bg_cls_weight
        self.matcher = HungarianMatcher3D(
            num_classes=num_classes,
            pc_range=pc_range,
            cls_weight=matcher_cls_weight,
            bbox_weight=matcher_bbox_weight,
            alpha=alpha,
            gamma=gamma,
        )

    def _loss_single(
        self,
        cls_scores: torch.Tensor,
        bbox_preds: torch.Tensor,
        gt_boxes: list[torch.Tensor],
        gt_labels: list[torch.Tensor],
    ) -> dict[str, torch.Tensor]:
        cls_scores = cls_scores.float()
        bbox_preds = bbox_preds.float()
        batch_size, num_queries, _ = cls_scores.shape
        assignments = self.matcher(cls_scores, bbox_preds, gt_boxes, gt_labels)

        label_indices = torch.full(
            (batch_size, num_queries), fill_value=self.num_classes, dtype=torch.long, device=cls_scores.device
        )
        bbox_targets = torch.zeros_like(bbox_preds)
        bbox_weights = torch.zeros_like(bbox_preds)

        num_pos = 0
        for batch_idx, (pred_ids, gt_ids) in enumerate(assignments):
            if pred_ids.numel() == 0:
                continue
            label_indices[batch_idx, pred_ids] = gt_labels[batch_idx][gt_ids]
            encoded_gt = encode_bbox_targets(gt_boxes[batch_idx].float()[gt_ids], self.pc_range).to(bbox_targets.dtype)
            bbox_targets[batch_idx, pred_ids] = encoded_gt
            bbox_weights[batch_idx, pred_ids] = 1.0
            num_pos += pred_ids.numel()

        num_total = batch_size * num_queries
        num_neg = num_total - num_pos
        normalizer = max(float(num_pos), 1.0)
        cls_avg_factor = max(float(num_pos) + self.bg_cls_weight * float(num_neg), 1.0)

        labels = torch.zeros((batch_size, num_queries, self.num_classes), dtype=cls_scores.dtype, device=cls_scores.device)
        pos_mask = label_indices != self.num_classes
        if pos_mask.any():
            batch_ids, query_ids = pos_mask.nonzero(as_tuple=True)
            labels[batch_ids, query_ids, label_indices[batch_ids, query_ids]] = 1.0

        pred_sigmoid = cls_scores.sigmoid()
        pt = pred_sigmoid * labels + (1 - pred_sigmoid) * (1 - labels)
        focal_weight = (self.alpha * labels + (1 - self.alpha) * (1 - labels)) * (1 - pt).pow(self.gamma)
        bce = F.binary_cross_entropy_with_logits(cls_scores, labels, reduction="none")
        loss_cls = (bce * focal_weight).sum() / cls_avg_factor * self.loss_cls_weight

        code_weights = self.code_weights.to(bbox_preds.device).view(1, 1, -1)
        abs_diff = (bbox_preds - bbox_targets).abs() * bbox_weights * code_weights
        loss_bbox = abs_diff.sum() / normalizer * self.loss_bbox_weight

        return {"loss_cls": loss_cls, "loss_bbox": loss_bbox}

    def loss_by_feat(
        self,
        all_cls_scores: torch.Tensor,
        all_bbox_preds: torch.Tensor,
        batch_gt_boxes: list[torch.Tensor],
        batch_gt_labels: list[torch.Tensor],
    ) -> dict[str, torch.Tensor]:
        if not self.use_auxiliary_losses:
            return self._loss_single(all_cls_scores[-1], all_bbox_preds[-1], batch_gt_boxes, batch_gt_labels)

        losses = [
            self._loss_single(all_cls_scores[layer_idx], all_bbox_preds[layer_idx], batch_gt_boxes, batch_gt_labels)
            for layer_idx in range(all_cls_scores.shape[0])
        ]

        output = {"loss_cls": losses[-1]["loss_cls"], "loss_bbox": losses[-1]["loss_bbox"]}
        for layer_idx in range(len(losses) - 1):
            output[f"d{layer_idx}.loss_cls"] = losses[layer_idx]["loss_cls"]
            output[f"d{layer_idx}.loss_bbox"] = losses[layer_idx]["loss_bbox"]
        return output

In [45]:
# --- Probe: finite scalar losses with gradient, on synthetic 2-layer predictions ---
bevformer_loss_probe = BEVFormerLoss(num_classes=3, pc_range=warp_pc_range)

loss_cls_scores = torch.randn(2, 1, 5, 3, requires_grad=True)   # [N_dec, B, Q, K]
loss_bbox_preds = torch.randn(2, 1, 5, 10, requires_grad=True)  # [N_dec, B, Q, 10]
loss_gt_boxes = [torch.tensor([[1.0, 2.0, 0.0, 2.0, 4.0, 1.5, 0.1, 0.0, 0.0]])]
loss_gt_labels = [torch.tensor([1])]

loss_dict = bevformer_loss_probe.loss_by_feat(loss_cls_scores, loss_bbox_preds, loss_gt_boxes, loss_gt_labels)
assert "loss_cls" in loss_dict and "loss_bbox" in loss_dict and "d0.loss_cls" in loss_dict
(loss_dict["loss_cls"] + loss_dict["loss_bbox"]).backward()
assert loss_cls_scores.grad is not None and loss_bbox_preds.grad is not None
print("BEVFormerLoss ok:", {k: round(v.item(), 4) for k, v in loss_dict.items()})

BEVFormerLoss ok: {'loss_cls': 4.8208, 'loss_bbox': 1.9551, 'd0.loss_cls': 7.3634, 'd0.loss_bbox': 2.2688}


## BEVFormerModel: The Full Forward Pass, Including Temporal History

Everything above composes into one model:

$$\text{images} \xrightarrow{\text{backbone+neck}} \text{mlvl\_feats} \xrightarrow{\text{encoder}} \text{bev\_embed} \xrightarrow{\text{decoder+head}} (\text{cls\_scores},\ \text{bbox\_preds})$$

But BEVFormer trains on a **queue** of $T$ consecutive frames, not a single
one, so temporal self-attention has real history to draw on. Official
BEVFormer's key memory/compute trick — reproduced exactly here — is to build
that history under `torch.no_grad()`:

$$\text{bev}_t = \begin{cases}
\text{Encoder}(\text{images}_0) & t = 0 \text{ (no history)} \\[4pt]
\text{Encoder}\big(\text{images}_t,\ \text{prev\_bev}=\operatorname{warp}(\text{bev}_{t-1},\ \Delta\xi_t)\big) & 0 < t < T-1 \quad \textbf{(no\_grad)} \\[4pt]
\text{Encoder}\big(\text{images}_{T-1},\ \text{prev\_bev}=\operatorname{warp}(\text{bev}_{T-2},\ \Delta\xi_{T-1})\big) & t = T-1 \quad \textbf{(grad enabled)}
\end{cases}$$

where $\Delta\xi_t = (\Delta x_t,\Delta y_t,\Delta\theta_t)$ is the ego-motion
delta between frames $t{-}1$ and $t$ (read from `can_bus`: the $(x,y)$
translation delta is stored directly at indices 16:18; the yaw delta is
recomputed on the fly from the absolute orientation quaternions stored at
indices 3:7 of consecutive frames, via $\psi = \operatorname{atan2}\big(2(wz+xy),\,1-2(y^2+z^2)\big)$).

Only $\text{bev}_{T-1}$ (the current frame) is ever fed through the decoder
and head — frames $0,\dots,T{-}2$ exist *purely* to seed temporal
self-attention's `prev_bev` input, so gradients through them would cost
memory for zero training benefit (the backbone/encoder weights they use are
already being trained through the current-frame path every step).

In [46]:
def _quaternion_to_yaw(quaternion: torch.Tensor) -> torch.Tensor:
    w, x, y, z = quaternion.unbind(-1)
    return torch.atan2(2 * (x * y + z * w), 1 - 2 * (y * y + z * z))


class BEVFormerModel(nn.Module):
    """Wires backbone, neck, BEV encoder, decoder, and head into one module."""

    def __init__(
        self,
        backbone: nn.Module,
        neck: nn.Module,
        encoder: nn.Module,
        decoder: nn.Module,
        head: nn.Module,
        grid_mask: nn.Module | None = None,
    ) -> None:
        super().__init__()
        self.backbone = backbone
        self.neck = neck
        self.encoder = encoder
        self.decoder = decoder
        self.head = head
        self.grid_mask = grid_mask

    def extract_bev_features(
        self,
        imgs: torch.Tensor,
        img_metas: list[dict],
        prev_bev: torch.Tensor | None = None,
        delta_translation_bev: torch.Tensor | None = None,
        delta_yaw: torch.Tensor | None = None,
    ) -> torch.Tensor:
        if self.grid_mask is not None and self.training:
            batch, num_cams, channels, height, width = imgs.shape
            flat = imgs.reshape(batch * num_cams, channels, height, width)
            flat = self.grid_mask(flat)
            imgs = flat.reshape(batch, num_cams, channels, height, width)

        features = self.backbone(imgs)
        pyramid = self.neck(features)
        mlvl_feats = list(pyramid.values())
        return self.encoder(
            mlvl_feats,
            img_metas,
            prev_bev=prev_bev,
            delta_translation_bev=delta_translation_bev,
            delta_yaw=delta_yaw,
        )

    def forward(
        self,
        imgs_queue: torch.Tensor,
        img_metas_queue: list[list[dict]],
        can_bus_queue: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        """
        Args:
            imgs_queue: [B, T, N, 3, H, W].
            img_metas_queue: list of length B, each a list of length T of dicts
                (matches `bevformer.data.collate.collate_fn`'s "img_metas" nesting).
            can_bus_queue: [B, T, 18].
        """
        batch, queue_length = imgs_queue.shape[:2]

        prev_bev = None
        for t in range(queue_length):
            img_metas_t = [img_metas_queue[b][t] for b in range(batch)]
            if t == 0:
                delta_translation_bev = None
                delta_yaw = None
            else:
                delta_translation_bev = can_bus_queue[:, t, 16:18]
                delta_yaw = _quaternion_to_yaw(can_bus_queue[:, t, 3:7]) - _quaternion_to_yaw(
                    can_bus_queue[:, t - 1, 3:7]
                )

            is_last_frame = t == queue_length - 1
            if is_last_frame:
                bev_embed = self.extract_bev_features(
                    imgs_queue[:, t], img_metas_t, prev_bev, delta_translation_bev, delta_yaw
                )
            else:
                with torch.no_grad():
                    prev_bev = self.extract_bev_features(
                        imgs_queue[:, t], img_metas_t, prev_bev, delta_translation_bev, delta_yaw
                    )

        hidden_states, _init_reference, inter_references = self.decoder(
            bev_embed, self.encoder.bev_h, self.encoder.bev_w, reference_point_predictor=self.head.predict_reference_points
        )
        cls_scores, bbox_preds = self.head(hidden_states, inter_references)
        return {"cls_scores": cls_scores, "bbox_preds": bbox_preds, "bev_embed": bev_embed}

In [47]:
class BEVFormerModel(nn.Module):
    """Wires backbone, neck, BEV encoder, decoder, and head into one module."""

    def __init__(
        self,
        backbone: nn.Module,
        neck: nn.Module,
        encoder: nn.Module,
        decoder: nn.Module,
        head: nn.Module,
        grid_mask: nn.Module | None = None,
    ) -> None:
        super().__init__()
        self.backbone = backbone
        self.neck = neck
        self.encoder = encoder
        self.decoder = decoder
        self.head = head
        self.grid_mask = grid_mask

    def extract_bev_features(
        self,
        imgs: torch.Tensor,
        img_metas: list[dict],
        prev_bev: torch.Tensor | None = None,
        delta_translation_bev: torch.Tensor | None = None,
        delta_yaw: torch.Tensor | None = None,
    ) -> torch.Tensor:
        if self.grid_mask is not None and self.training:
            batch, num_cams, channels, height, width = imgs.shape
            flat = imgs.reshape(batch * num_cams, channels, height, width)
            flat = self.grid_mask(flat)
            imgs = flat.reshape(batch, num_cams, channels, height, width)

        features = self.backbone(imgs)
        pyramid = self.neck(features)
        mlvl_feats = list(pyramid.values())
        return self.encoder(
            mlvl_feats,
            img_metas,
            prev_bev=prev_bev,
            delta_translation_bev=delta_translation_bev,
            delta_yaw=delta_yaw,
        )

    def forward(
        self,
        imgs_queue: torch.Tensor,
        img_metas_queue: list[list[dict]],
        can_bus_queue: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        """
        Args:
            imgs_queue: [B, T, N, 3, H, W].
            img_metas_queue: list of length B, each a list of length T of dicts
                (matches `bevformer.data.collate.collate_fn`'s "img_metas" nesting).
            can_bus_queue: [B, T, 18].
        """
        batch, queue_length = imgs_queue.shape[:2]

        prev_bev = None
        for t in range(queue_length):
            img_metas_t = [img_metas_queue[b][t] for b in range(batch)]
            if t == 0:
                delta_translation_bev = None
                delta_yaw = None
            else:
                delta_translation_bev = can_bus_queue[:, t, 16:18]
                delta_yaw = _quaternion_to_yaw(can_bus_queue[:, t, 3:7]) - _quaternion_to_yaw(
                    can_bus_queue[:, t - 1, 3:7]
                )

            is_last_frame = t == queue_length - 1
            if is_last_frame:
                bev_embed = self.extract_bev_features(
                    imgs_queue[:, t], img_metas_t, prev_bev, delta_translation_bev, delta_yaw
                )
            else:
                with torch.no_grad():
                    prev_bev = self.extract_bev_features(
                        imgs_queue[:, t], img_metas_t, prev_bev, delta_translation_bev, delta_yaw
                    )

        hidden_states, _init_reference, inter_references = self.decoder(
            bev_embed, self.encoder.bev_h, self.encoder.bev_w, reference_point_predictor=self.head.predict_reference_points
        )
        cls_scores, bbox_preds = self.head(hidden_states, inter_references)
        return {"cls_scores": cls_scores, "bbox_preds": bbox_preds, "bev_embed": bev_embed}

In [48]:
# --- Probe: full model, 3-frame queue, 2 cameras, tiny everything ---
def build_probe_model(bev_h=4, bev_w=4, embed_dims=16, num_cams=2, num_classes=3, num_queries=6):
    backbone = MultiViewImageBackbone(variant="resnet50", pretrained=False, frozen_stages=-1)
    neck = ImageFPN(in_channels=(512, 1024, 2048), out_channels=embed_dims, out_names=("p3", "p4", "p5", "p6"))
    encoder = BEVFormerEncoder(
        num_layers=1, bev_h=bev_h, bev_w=bev_w, embed_dims=embed_dims, pc_range=warp_pc_range,
        num_cams=num_cams, num_heads=4, num_levels=4, num_points_in_pillar=2,
        num_points_temporal=2, feedforward_dims=32,
    )
    decoder = BEVFormerDecoder(embed_dims=embed_dims, num_queries=num_queries, num_layers=2, num_heads=4, num_points=2, ffn_channels=32)
    head = BEVFormerHead(embed_dims=embed_dims, num_classes=num_classes, box_dim=10, num_decoder_layers=2, pc_range=warp_pc_range)
    return BEVFormerModel(backbone, neck, encoder, decoder, head)


model_probe = build_probe_model()
model_batch, model_queue_len, model_num_cams = 1, 3, 2
model_imgs_queue = torch.randn(model_batch, model_queue_len, model_num_cams, 3, 64, 64)
model_img_metas_queue = [
    [{"lidar2img": [identity_lidar2img(), identity_lidar2img()], "image_size": (64, 64)} for _ in range(model_queue_len)]
    for _ in range(model_batch)
]
model_can_bus_queue = torch.zeros(model_batch, model_queue_len, 18)

model_outputs = model_probe(model_imgs_queue, model_img_metas_queue, model_can_bus_queue)
assert model_outputs["cls_scores"].shape == (2, 1, 6, 3)
assert model_outputs["bbox_preds"].shape == (2, 1, 6, 10)
assert model_outputs["bev_embed"].shape == (1, 16, 16)

(model_outputs["cls_scores"].sum() + model_outputs["bbox_preds"].sum()).backward()
for name, param in model_probe.backbone.named_parameters():
    if param.requires_grad:
        assert param.grad is not None and torch.isfinite(param.grad).all(), name

print("BEVFormerModel end-to-end ok:")
print("  cls_scores:", tuple(model_outputs["cls_scores"].shape), " # [N_dec, B, Q, K]")
print("  bbox_preds:", tuple(model_outputs["bbox_preds"].shape), " # [N_dec, B, Q, 10]")
print("  bev_embed :", tuple(model_outputs["bev_embed"].shape), " # [B, H_bev*W_bev, C] (last frame only)")
print("  backbone gradient reached, confirming only the last frame's forward pass was tracked for autograd")

BEVFormerModel end-to-end ok:
  cls_scores: (2, 1, 6, 3)  # [N_dec, B, Q, K]
  bbox_preds: (2, 1, 6, 10)  # [N_dec, B, Q, 10]
  bev_embed : (1, 16, 16)  # [B, H_bev*W_bev, C] (last frame only)
  backbone gradient reached, confirming only the last frame's forward pass was tracked for autograd


## End-To-End Shape Evolution

Tracing one training step of the probe model above through every stage
(`B=1, T=3, N=2, C=16, H_bev=W_bev=4, Q=6, K=3, N_dec=2`):

| Stage | Output shape | Meaning |
|---|---|---|
| `imgs_queue` | `[1, 3, 2, 3, 64, 64]` | `[B, T, N, 3, H, W]` raw queue input |
| `backbone(imgs[t])` | `{stage3: [1,2,512,8,8], stage4: [1,2,1024,4,4], stage5: [1,2,2048,2,2]}` | one frame's per-camera multi-stage features |
| `neck(...)` | `{p3..p6}`, each `[1,2,16,H_l,W_l]` | 4-level pyramid, channel-unified to `C=16` |
| `encoder(mlvl_feats, prev_bev)` | `[1, 16, 16]` | `bev_embed`: `[B, H_bev*W_bev, C]`, one per frame (only last one keeps gradient) |
| `decoder(bev_embed)` | `hidden_states: [2, 1, 6, 16]` | `[N_dec, B, Q, C]` |
| `head(hidden_states, ...)` | `cls_scores: [2,1,6,3]`, `bbox_preds: [2,1,6,10]` | per-decoder-layer predictions |
| `loss.loss_by_feat(...)` | scalars (`loss_cls`, `loss_bbox`, `d0.loss_cls`, `d0.loss_bbox`) | final + auxiliary supervision |

## Summary

This notebook re-derived, from first principles and without importing the
production package, every non-trivial piece of BEVFormer used by this
repository: a shared multi-camera backbone with learned deformable
convolutions, a 4-level FPN, the two attentions that make BEVFormer
*BEVFormer* (deformable spatial cross-attention lifting images into a BEV
grid, and deformable temporal self-attention fusing a warped previous frame),
an object-query decoder reusing the same deformable-attention core, DETR-style
Hungarian matching and focal/L1 losses with auxiliary supervision, and the
no-grad temporal-history mechanism that ties frames together during training.

The production implementation under `bevformer/` is unit-tested per-module
(see `tests/`); this notebook is the complementary, narrative counterpart —
useful for building intuition, onboarding, or simply checking that "the math
in the paper" and "the code in this repo" are the same thing.